In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

def summarize_telemetry(csv_path, name_col=None, top_cmdb=20, top_names=50):
    # Load
    df = pd.read_csv(csv_path)
    # Ensure timestamp and cmdb_id exist
    if 'timestamp' not in df.columns:
        raise ValueError(f"{csv_path} missing 'timestamp' column")
    if 'cmdb_id' not in df.columns:
        raise ValueError(f"{csv_path} missing 'cmdb_id' column")
    # Convert timestamp to timezone-aware datetimes (UTC -> Asia/Shanghai)
    df['ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    total_rows = len(df)
    min_ts = df['ts_dt'].min()
    max_ts = df['ts_dt'].max()
    distinct_cmdb_count = df['cmdb_id'].nunique()
    # Top cmdb_id by row count (limit)
    top_cmdb_df = df['cmdb_id'].value_counts().reset_index()
    top_cmdb_df.columns = ['cmdb_id', 'count']
    top_cmdb_df = top_cmdb_df.head(top_cmdb).reset_index(drop=True)
    # If name_col provided, compute top names; else produce empty DataFrame
    if name_col and name_col in df.columns:
        top_names_df = df[name_col].value_counts().reset_index()
        top_names_df.columns = [name_col, 'count']
        top_names_df = top_names_df.head(top_names).reset_index(drop=True)
        distinct_name_count = df[name_col].nunique()
    else:
        top_names_df = pd.DataFrame(columns=[name_col or 'name', 'count'])
        distinct_name_count = 0
    # Compact one-row summary DataFrame
    summary_df = pd.DataFrame([{
        'file': csv_path,
        'total_rows': total_rows,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'distinct_cmdb_ids': distinct_cmdb_count,
        'distinct_names': distinct_name_count
    }])
    return df, summary_df, top_cmdb_df, top_names_df

# Summarize each file according to requested name columns
df_metric, metric_summary, metric_top_cmdb, metric_top_kpi = summarize_telemetry('metric.csv', name_col='kpi_name', top_cmdb=20, top_names=50)
df_trace, trace_summary, trace_top_cmdb, trace_top_trace = summarize_telemetry('trace.csv', name_col='trace_name', top_cmdb=20, top_names=50)
df_log, log_summary, log_top_cmdb, log_top_log = summarize_telemetry('log.csv', name_col='log_name', top_cmdb=20, top_names=50)
df_error, error_summary, error_top_cmdb, error_top_message = summarize_telemetry('error_logs.csv', name_col='message', top_cmdb=20, top_names=50)

# Display compact results (each variable is concise: single-row summaries and top-N lists)
metric_summary, metric_top_cmdb, metric_top_kpi, trace_summary, trace_top_cmdb, trace_top_trace, log_summary, log_top_cmdb, log_top_log, error_summary, error_top_cmdb, error_top_message

```
Out[1]:
```
```python
# Build a concise plain-English summary from previously computed summaries and top-N tables.
s_parts = []

# Metric summary
m = metric_summary.iloc[0]
m_top_cmdb = ", ".join([f"{r['cmdb_id']}({r['count']})" for _, r in metric_top_cmdb.head(5).iterrows()])
m_top_name_examples = ", ".join([f"{r['kpi_name'].split()[0][:60]}...({r['count']})" for _, r in metric_top_kpi.head(3).iterrows()])
s_parts.append(
    f"metric.csv: {m['total_rows']} rows; time range {m['min_timestamp']} to {m['max_timestamp']}; "
    f"{m['distinct_cmdb_ids']} distinct cmdb_id values (top5: {m_top_cmdb}); "
    f"{m['distinct_names']} distinct kpi_name values (example top3: {m_top_name_examples})."
)

# Trace summary
t = trace_summary.iloc[0]
t_top_cmdb = ", ".join([f"{r['cmdb_id']}({r['count']})" for _, r in trace_top_cmdb.head(5).iterrows()])
t_top_name_examples = ", ".join([f"{r['trace_name'].split()[0][:60]}...({r['count']})" for _, r in trace_top_trace.head(3).iterrows()])
s_parts.append(
    f"trace.csv: {t['total_rows']} rows; time range {t['min_timestamp']} to {t['max_timestamp']}; "
    f"{t['distinct_cmdb_ids']} distinct cmdb_id values (top5: {t_top_cmdb}); "
    f"{t['distinct_names']} distinct trace_name values (example top3: {t_top_name_examples})."
)

# Log summary
l = log_summary.iloc[0]
l_top_cmdb = ", ".join([f"{r['cmdb_id']}({r['count']})" for _, r in log_top_cmdb.head(5).iterrows()])
l_top_name_examples = ", ".join([f"{r['log_name']}({r['count']})" for _, r in log_top_log.head(3).iterrows()])
s_parts.append(
    f"log.csv: {l['total_rows']} rows; time range {l['min_timestamp']} to {l['max_timestamp']}; "
    f"{l['distinct_cmdb_ids']} distinct cmdb_id values (top5: {l_top_cmdb}); "
    f"{l['distinct_names']} distinct log_name values (top: {l_top_name_examples})."
)

# Error logs summary
e = error_summary.iloc[0]
e_top_cmdb = ", ".join([f"{r['cmdb_id']}({r['count']})" for _, r in error_top_cmdb.head(5).iterrows()])
e_top_msg_examples = ", ".join([f"{r['message'][:80].replace('\\n',' ')}...({r['count']})" for _, r in error_top_message.head(1).iterrows()])
s_parts.append(
    f"error_logs.csv: {e['total_rows']} rows; time range {e['min_timestamp']} to {e['max_timestamp']}; "
    f"{e['distinct_cmdb_ids']} distinct cmdb_id values (counts: {e_top_cmdb}); "
    f"{e['distinct_names']} distinct messages (top: {e_top_msg_examples})."
)

summary = "\n".join(s_parts)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         file  total_rows             min_timestamp             max_timestamp  distinct_cmdb_ids  distinct_names
0  metric.csv      152352 2022-03-21 21:30:00+08:00 2022-03-21 21:59:00+08:00                 59            1292,                      cmdb_id  count
0                  adservice  10230
1                 adservice2   9990
2                frontend2-0   5520
3                 frontend-2   5400
4                 frontend-1   5400
5                 frontend-0   5400
6          checkoutservice-0   4260
7          checkoutservice-2   4140
8          checkoutservice-1   3900
9         checkoutservice2-0   3660
10   productcatalogservice-1   3480
11   productcatalogservice-2   3480
12   productcatalogservice-0   3480
13              adservice2-0   3300
14            cartservice2-0   3000
15  productcatalogservice2-0   2880
16             cartservice-1   2880
17               adservice-2   2820
18               adservice-1   2820
19             cartservice-0   2760,                                              kpi_name  count
0   container.node-5.container_network_receive_MB....   1050
1   container.node-5.container_network_receive_err...   1050
2   container.node-5.container_network_receive_pac...   1050
3   container.node-5.container_network_receive_pac...   1050
4   container.node-5.container_network_transmit_MB...   1050
..                                                ...    ...
45       container.node-5.container_cpu_usage_seconds    990
46      container.node-5.container_cpu_system_seconds    990
47  container.node-5.container_fs_reads_merged./de...    990
48  container.node-5.container_fs_sector_reads./de...    990
49      container.node-5.container_memory_mapped_file    990

[50 rows x 2 columns],         file  total_rows             min_timestamp             max_timestamp  distinct_cmdb_ids  distinct_names
0  trace.csv       31536 2022-03-21 21:30:00+08:00 2022-03-21 21:59:00+08:00                 40             200,                     cmdb_id  count
0                frontend-1   2608
1                frontend-2   2592
2                frontend-0   2412
3         checkoutservice-0   1956
4         checkoutservice-2   1908
5         checkoutservice-1   1784
6   productcatalogservice-0   1116
7   productcatalogservice-2   1112
8   productcatalogservice-1   1096
9               frontend2-0   1080
10       checkoutservice2-0   1080
11  recommendationservice-0    840
12  recommendationservice-2    840
13  recommendationservice-1    840
14            cartservice-1    752
15            cartservice-2    752
16            cartservice-0    744
17        currencyservice-2    668
18        currencyservice-0    668
19        currencyservice-1    664,                                   trace_name  count
0            trace.from_frontend-1.row_count    592
1           trace.from_frontend-1.error_rate    592
2         trace.from_frontend-1.duration_p95    592
3        trace.from_frontend-1.duration_mean    592
4            trace.from_frontend-2.row_count    588
..                                       ...    ...
45       trace.from_frontend2-0.duration_p95    210
46         trace.from_frontend2-0.error_rate    210
47          trace.from_frontend2-0.row_count    210
48   trace.from_checkoutservice2-0.row_count    180
49  trace.from_checkoutservice2-0.error_rate    180

[50 rows x 2 columns],       file  total_rows             min_timestamp             max_timestamp  distinct_cmdb_ids  distinct_names
0  log.csv        1750 2022-03-21 21:30:00+08:00 2022-03-21 21:59:00+08:00                 31               2,                     cmdb_id  count
0               adservice-0     60
1               adservice-1     60
2               adservice-2     60
3             cartservice-0     60
4             cartservice-1     60
5             cartservice-2     60
6         currencyservice-1     60
7         currencyservice-2     60
8         currencyservice-0     60
9   recommendationservice-1     60
10  productcatalogservice-0     60
11  productcatalogservice-1     60
12               frontend-2     60
13               frontend-1     60
14               frontend-0     60
15        shippingservice-0     60
16        shippingservice-1     60
17        shippingservice-2     60
18  recommendationservice-2     60
19  recommendationservice-0     60,           log_name  count
0  log.error_count    875
1    log.row_count    875,              file  total_rows             min_timestamp             max_timestamp  distinct_cmdb_ids  distinct_names
0  error_logs.csv         743 2022-03-21 21:30:00+08:00 2022-03-21 21:59:55+08:00                  3               1,       cmdb_id  count
0  frontend-2    334
1  frontend-1    272
2  frontend-0    137,                                              message  count
0  severity: warning, message: failed to retrieve...    743)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse df_metric and tz if present; otherwise load metric.csv and prepare timestamps
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    tz = pytz.timezone('Asia/Shanghai')
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure ts_dt exists and is timezone-aware in Asia/Shanghai
if 'ts_dt' not in df_metric.columns:
    tz = pytz.timezone('Asia/Shanghai')
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz variable exists
    try:
        tz
    except NameError:
        tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using full series (before filtering)
thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], sort=False)['value']
    .agg(p95=lambda x: float(x.quantile(0.95)), p5=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# 2) Filter to incident window (UTC+8)
start = tz.localize(pd.Timestamp('2022-03-21 21:30:00'))
end = tz.localize(pd.Timestamp('2022-03-21 22:00:00'))
df_win = df_metric[(df_metric['ts_dt'] >= start) & (df_metric['ts_dt'] <= end)].copy()

# Merge thresholds into windowed data so each row has p95/p5 for its series
df_m = df_win.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Flag anomalies according to global thresholds
df_m['high'] = df_m['value'] >= df_m['p95']
df_m['low'] = df_m['value'] <= df_m['p5']
df_m['is_anom'] = df_m['high'] | df_m['low']

# Aggregate per (cmdb_id, kpi_name)
def summarize_group(g):
    anom_count = int(g['is_anom'].sum())
    earliest = g.loc[g['is_anom'], 'ts_dt'].min() if anom_count > 0 else pd.NaT
    any_high = bool(g['high'].any())
    any_low = bool(g['low'].any())
    direction = ('both' if (any_high and any_low) else ('high' if any_high else ('low' if any_low else 'none')))
    return pd.Series({
        'p95': float(g['p95'].iloc[0]),
        'p5': float(g['p5'].iloc[0]),
        'anomaly_count': anom_count,
        'anomaly_direction': direction,
        'earliest_anomaly_timestamp': earliest,
        'max_value_in_window': float(g['value'].max()) if len(g)>0 else np.nan,
        'min_value_in_window': float(g['value'].min()) if len(g)>0 else np.nan
    })

summaries = df_m.groupby(['cmdb_id', 'kpi_name'], sort=False).apply(summarize_group).reset_index()

# Keep only series with at least one anomaly in the window
summaries = summaries[summaries['anomaly_count'] > 0].copy()

# Format earliest_anomaly_timestamp as ISO-like string with timezone for compactness
summaries['earliest_anomaly_timestamp'] = summaries['earliest_anomaly_timestamp'].apply(
    lambda x: x.isoformat() if pd.notnull(x) else ''
)

# 3) Return top 20 pairs sorted by anomaly_count desc with requested columns
out_cols = ['cmdb_id', 'kpi_name', 'p95', 'p5', 'anomaly_count', 'anomaly_direction',
            'earliest_anomaly_timestamp', 'max_value_in_window', 'min_value_in_window']

top20 = summaries.sort_values(by='anomaly_count', ascending=False).head(20)[out_cols].reset_index(drop=True)

# Keep numeric columns compact (round p95/p5 and values)
top20['p95'] = top20['p95'].round(6)
top20['p5'] = top20['p5'].round(6)
top20['max_value_in_window'] = top20['max_value_in_window'].round(6)
top20['min_value_in_window'] = top20['min_value_in_window'].round(6)

top20

```
Out[2]:
```
```python
# Build a concise plain-English summary from the previously computed top20/summaries variables.
try:
    top20
except NameError:
    # If top20 isn't available, try to load results minimally (fallback)
    import pandas as pd
    df_metric = pd.read_csv('metric.csv')
    tz = pytz.timezone('Asia/Shanghai')
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    summary = "Top-20 results not available in memory; please re-run the anomaly aggregation."
else:
    # Compute some aggregate facts
    total_shown = top20.shape[0]
    unique_cmdbs = top20['cmdb_id'].nunique()
    cmdb_list = top20['cmdb_id'].unique().tolist()
    # Check common anomaly_count, directions, earliest timestamp pattern
    common_anom_count = int(top20['anomaly_count'].mode()[0]) if 'anomaly_count' in top20 else None
    common_direction = top20['anomaly_direction'].mode()[0] if 'anomaly_direction' in top20 else None
    earliest_times = top20['earliest_anomaly_timestamp'].unique().tolist()
    # Inspect whether p95 == p5 == observed values for many rows
    equal_thresholds = (top20['p95'] == top20['p5']).sum()
    equal_values_to_p95 = (top20['max_value_in_window'] == top20['p95']).sum()
    # Build summary text
    summary_lines = [
        f"Top {total_shown} anomalous (cmdb_id, kpi_name) pairs (sorted by anomaly count):",
        f"- All {total_shown} shown pairs belong to {unique_cmdbs} cmdb_id(s): {', '.join(cmdb_list)}.",
        f"- Each shown pair has anomaly_count = {common_anom_count} (i.e., {common_anom_count} points in the 30-minute window).",
        f"- Anomaly direction for the shown pairs is '{common_direction}' (most rows flagged as '{common_direction}').",
        f"- Earliest anomaly timestamps in the shown rows start at: {', '.join(sorted(set(earliest_times)))}.",
        f"- For {equal_thresholds} of the {total_shown} rows p95 equals p5 (no spread); for {equal_values_to_p95} rows the window max equals p95.",
        "",
        "Interpretation:",
        "- The top anomalous series are all metrics named like 'container.node-5.*' for cmdb_id 'shippingservice2-0'.",
        "- Each series shows 30 anomaly points across the 30-minute incident window (one per minute), with earliest at the window start.",
        "- Many of these KPIs are constant (p95 == p5 == observed value), causing them to be flagged as both 'high' and 'low' against global thresholds.",
        "- These appear to be static/spec/config or non-varying metrics (e.g., spec limits, start time, fixed counts), so their anomalies likely reflect constant values rather than meaningful runtime deviations.",
        "",
        "Recommendation:",
        "- Exclude constant/config metrics (where p95==p5==value) from anomaly-focused root-cause lists, and focus on variable runtime KPIs for troubleshooting."
    ]
    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name           p95            p5  anomaly_count anomaly_direction earliest_anomaly_timestamp  max_value_in_window  min_value_in_window
0   shippingservice2-0    container.node-5.container_spec_memory_limit_MB  1.280000e+02  1.280000e+02             30              both  2022-03-21T21:30:00+08:00         1.280000e+02         1.280000e+02
1   shippingservice2-0  container.node-5.container_spec_memory_reserva...  0.000000e+00  0.000000e+00             30              both  2022-03-21T21:30:00+08:00         0.000000e+00         0.000000e+00
2   shippingservice2-0  container.node-5.container_spec_memory_swap_li...  1.280000e+02  1.280000e+02             30              both  2022-03-21T21:30:00+08:00         1.280000e+02         1.280000e+02
3   shippingservice2-0      container.node-5.container_start_time_seconds  1.647523e+09  1.647523e+09             30              both  2022-03-21T21:30:00+08:00         1.647523e+09         1.647523e+09
4   shippingservice2-0   container.node-5.container_tasks_state.iowaiting  0.000000e+00  0.000000e+00             30              both  2022-03-21T21:30:00+08:00         0.000000e+00         0.000000e+00
5   shippingservice2-0     container.node-5.container_tasks_state.running  0.000000e+00  0.000000e+00             30              both  2022-03-21T21:30:00+08:00         0.000000e+00         0.000000e+00
6   shippingservice2-0    container.node-5.container_tasks_state.sleeping  0.000000e+00  0.000000e+00             30              both  2022-03-21T21:30:00+08:00         0.000000e+00         0.000000e+00
7   shippingservice2-0     container.node-5.container_tasks_state.stopped  0.000000e+00  0.000000e+00             30              both  2022-03-21T21:30:00+08:00         0.000000e+00         0.000000e+00
8   shippingservice2-0  container.node-5.container_network_transmit_er...  0.000000e+00  0.000000e+00             30              both  2022-03-21T21:30:00+08:00         0.000000e+00         0.000000e+00
9   shippingservice2-0  container.node-5.container_network_transmit_pa...  0.000000e+00  0.000000e+00             30              both  2022-03-21T21:30:00+08:00         0.000000e+00         0.000000e+00
10  shippingservice2-0                 container.node-5.container_sockets  4.000000e+00  4.000000e+00             30              both  2022-03-21T21:30:00+08:00         4.000000e+00         4.000000e+00
11  shippingservice2-0         container.node-5.container_spec_cpu_period  1.000000e+05  1.000000e+05             30              both  2022-03-21T21:30:00+08:00         1.000000e+05         1.000000e+05
12  shippingservice2-0          container.node-5.container_spec_cpu_quota  2.000000e+04  2.000000e+04             30              both  2022-03-21T21:30:00+08:00         2.000000e+04         2.000000e+04
13  shippingservice2-0         container.node-5.container_spec_cpu_shares  1.020000e+02  1.020000e+02             30              both  2022-03-21T21:30:00+08:00         1.020000e+02         1.020000e+02
14  shippingservice2-0             container.node-5.container_memory_swap  0.000000e+00  0.000000e+00             30              both  2022-03-21T21:30:00+08:00         0.000000e+00         0.000000e+00
15  shippingservice2-0  container.node-5.container_network_receive_err...  0.000000e+00  0.000000e+00             30              both  2022-03-21T21:30:00+08:00         0.000000e+00         0.000000e+00
16  shippingservice2-0  container.node-5.container_network_receive_pac...  0.000000e+00  0.000000e+00             30              both  2022-03-21T21:30:00+08:00         0.000000e+00         0.000000e+00
17  shippingservice2-0            container.node-5.container_memory_cache  2.181120e+07  2.181120e+07             30              both  2022-03-21T21:30:00+08:00         2.181120e+07         2.181120e+07
18  shippingservice2-0          container.node-5.container_memory_failcnt  2.399892e+06  2.399892e+06             30              both  2022-03-21T21:30:00+08:00         2.399892e+06         2.399892e+06
19  shippingservice2-0  container.node-5.container_memory_failures.con...  0.000000e+00  0.000000e+00             30              both  2022-03-21T21:30:00+08:00         0.000000e+00         0.000000e+00```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric and tz if available, otherwise load and prepare
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    tz = pytz.timezone('Asia/Shanghai')
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure ts_dt exists and timezone is set to Asia/Shanghai
    try:
        tz
    except NameError:
        tz = pytz.timezone('Asia/Shanghai')
    if 'ts_dt' not in df_metric.columns:
        df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        # ensure tz-aware (convert naive UTC->Asia/Shanghai if needed)
        if df_metric['ts_dt'].dt.tz is None:
            df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using the FULL series (required by rule 9)
thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], sort=False)['value']
    .agg(p95=lambda x: float(x.quantile(0.95)), p5=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# 2) Filter to incident window (UTC+8) 2022-03-21 21:30:00+08:00 through 2022-03-21 21:59:00+08:00
start = tz.localize(pd.Timestamp('2022-03-21 21:30:00'))
end = tz.localize(pd.Timestamp('2022-03-21 21:59:00'))
df_win = df_metric[(df_metric['ts_dt'] >= start) & (df_metric['ts_dt'] <= end)].copy()

# Merge thresholds (global) into the windowed rows
df_m = df_win.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Flag anomalies using the global thresholds
df_m['high'] = df_m['value'] >= df_m['p95']
df_m['low'] = df_m['value'] <= df_m['p5']
df_m['is_anom'] = df_m['high'] | df_m['low']

# 2) Aggregate per (cmdb_id, kpi_name) in the window
def agg_window(g):
    anom_count = int(g['is_anom'].sum())
    earliest = g.loc[g['is_anom'], 'ts_dt'].min() if anom_count > 0 else pd.NaT
    any_high = bool(g['high'].any())
    any_low = bool(g['low'].any())
    direction = ('both' if (any_high and any_low) else ('high' if any_high else ('low' if any_low else 'none')))
    return pd.Series({
        'p95': float(g['p95'].iloc[0]),
        'p5': float(g['p5'].iloc[0]),
        'anomaly_count': anom_count,
        'anomaly_direction': direction,
        'earliest_anomaly_timestamp': earliest,
        'max_value_in_window': float(g['value'].max()) if len(g) > 0 else np.nan,
        'min_value_in_window': float(g['value'].min()) if len(g) > 0 else np.nan,
        'unique_values_count_in_window': int(g['value'].nunique())
    })

summaries = df_m.groupby(['cmdb_id', 'kpi_name'], sort=False).apply(agg_window).reset_index()

# 3) Apply filters:
#   (1) p95 != p5
#   (2) max_value_in_window != min_value_in_window
#   (3) anomaly_count > 0
filtered = summaries[
    (summaries['p95'] != summaries['p5']) &
    (summaries['max_value_in_window'] != summaries['min_value_in_window']) &
    (summaries['anomaly_count'] > 0)
].copy()

# Format earliest timestamp and round numeric columns for compactness
filtered['earliest_anomaly_timestamp'] = filtered['earliest_anomaly_timestamp'].apply(
    lambda x: x.isoformat() if pd.notnull(x) else ''
)
for c in ['p95', 'p5', 'max_value_in_window', 'min_value_in_window']:
    filtered[c] = filtered[c].round(6)

out_cols = ['cmdb_id', 'kpi_name', 'p95', 'p5', 'anomaly_count', 'anomaly_direction',
            'earliest_anomaly_timestamp', 'max_value_in_window', 'min_value_in_window',
            'unique_values_count_in_window']

top20_filtered = filtered.sort_values(by='anomaly_count', ascending=False).head(20)[out_cols].reset_index(drop=True)

top20_filtered

```
Out[3]:
```
```python
# Compose a concise plain-English summary from the top20_filtered results
try:
    top20_filtered
except NameError:
    summary = "No precomputed results available in memory. Please run the anomaly aggregation step first."
else:
    total = top20_filtered.shape[0]
    unique_cmdbs = top20_filtered['cmdb_id'].nunique()
    cmdb_sample = ", ".join(sorted(top20_filtered['cmdb_id'].unique())[:6])
    common_anom_count = int(top20_filtered['anomaly_count'].mode()[0]) if 'anomaly_count' in top20_filtered else None
    common_direction = top20_filtered['anomaly_direction'].mode()[0] if 'anomaly_direction' in top20_filtered else None
    earliest_times = sorted(top20_filtered['earliest_anomaly_timestamp'].unique())
    kpi_examples = ", ".join(top20_filtered['kpi_name'].head(6).tolist())
    avg_unique_vals = top20_filtered['unique_values_count_in_window'].mean()
    
    summary_lines = [
        f"Top {total} (cmdb_id, kpi_name) pairs passing the filters (p95!=p5, max!=min, anomaly_count>0):",
        f"- These involve {unique_cmdbs} distinct cmdb_id(s); example cmdb_ids: {cmdb_sample}.",
        f"- Each shown pair has anomaly_count = {common_anom_count} (i.e., {common_anom_count} points in the 30-min window).",
        f"- Most pairs are labeled anomaly_direction = '{common_direction}'.",
        f"- Earliest anomalies for these pairs occur at: {', '.join(earliest_times)} (UTC+8).",
        f"- Example KPIs among the top rows: {kpi_examples}.",
        f"- The window unique value counts are small (average ≈ {avg_unique_vals:.1f}), typically 2–3 distinct values per series in-window.",
        "",
        "Interpretation and guidance:",
        "- The anomalies are concentrated across container-level KPIs (memory_mapped_file, file_descriptors, sockets, fs_writes, memory usage, etc.) for multiple services.",
        "- Each series shows persistent anomalies for every minute of the window (30 points), often as small discrete value changes rather than wide continuous drift.",
        "- Because p95 != p5 and max!=min, these are not strictly constant metrics, but the small number of unique values suggests step changes or quantized counters.",
        "- Next steps: focus troubleshooting on the listed services and KPIs (e.g., memory_mapped_file, file_descriptors, sockets, FS writes) and check container/node-level events or deployments around 2022-03-21 21:30+08:00."
    ]
    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name           p95            p5  anomaly_count anomaly_direction earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  unique_values_count_in_window
0          shippingservice-2      container.node-5.container_memory_mapped_file  1.933312e+06  1.925120e+06             30              both  2022-03-21T21:30:00+08:00         1.933312e+06         1.925120e+06                              2
1          paymentservice2-0      container.node-6.container_memory_mapped_file  5.593088e+05  5.120000e+05             30              both  2022-03-21T21:30:00+08:00         5.980160e+05         5.120000e+05                              2
2           paymentservice-2                 container.node-5.container_sockets  3.000000e+00  2.500000e+00             30              both  2022-03-21T21:30:00+08:00         3.000000e+00         2.500000e+00                              2
3          checkoutservice-2                 container.node-5.container_sockets  3.000000e+00  2.500000e+00             30              both  2022-03-21T21:30:00+08:00         3.000000e+00         2.500000e+00                              2
4          checkoutservice-2  mesh.source.checkoutservice.jaeger-collector.i...  2.890000e-04  0.000000e+00             30              both  2022-03-21T21:30:00+08:00         5.250000e-04         0.000000e+00                              3
5          checkoutservice-2        container.node-5.container_file_descriptors  9.000000e+00  8.500000e+00             30              both  2022-03-21T21:30:00+08:00         9.000000e+00         8.500000e+00                              2
6          paymentservice2-0        container.node-6.container_file_descriptors  2.200000e+01  2.145000e+01             30              both  2022-03-21T21:30:00+08:00         2.200000e+01         2.100000e+01                              2
7             emailservice-0                 container.node-5.container_sockets  4.000000e+00  3.500000e+00             30              both  2022-03-21T21:30:00+08:00         4.000000e+00         3.500000e+00                              2
8             emailservice-0        container.node-5.container_file_descriptors  1.000000e+01  9.500000e+00             30              both  2022-03-21T21:30:00+08:00         1.000000e+01         9.500000e+00                              2
9         checkoutservice2-0  mesh.source.checkoutservice2.jaeger-collector....  5.250000e-04  0.000000e+00             30              both  2022-03-21T21:30:00+08:00         1.050000e-03         0.000000e+00                              3
10                frontend-2   container.node-5.container_fs_writes_MB./dev/vda  1.503900e-02  0.000000e+00             30              both  2022-03-21T21:30:00+08:00         8.593800e-02         0.000000e+00                              3
11               adservice-1   container.node-5.container_memory_working_set_MB  1.987385e+02  1.987148e+02             30              both  2022-03-21T21:30:00+08:00         1.988008e+02         1.987148e+02                              3
12               adservice-1                 container.node-5.container_threads  2.327500e+01  2.300000e+01             30              both  2022-03-21T21:30:00+08:00         2.400000e+01         2.300000e+01                              3
13               adservice-2        container.node-5.container_file_descriptors  9.800000e+01  9.700000e+01             30              both  2022-03-21T21:30:00+08:00         9.800000e+01         9.700000e+01                              2
14  productcatalogservice2-0            container.node-6.container_memory_cache  1.155482e+07  1.153434e+07             30              both  2022-03-21T21:30:00+08:00         1.155482e+07         1.153434e+07                              2
15  productcatalogservice2-0      container.node-6.container_memory_mapped_file  5.939200e+05  5.898240e+05             30              both  2022-03-21T21:30:00+08:00         5.939200e+05         5.898240e+05                              2
16               adservice-1         container.node-5.container_memory_usage_MB  1.987893e+02  1.987656e+02             30              both  2022-03-21T21:30:00+08:00         1.988516e+02         1.987656e+02                              3
17               adservice-1              container.node-5.container_memory_rss  1.433680e+08  1.433477e+08             30              both  2022-03-21T21:30:00+08:00         1.434214e+08         1.433477e+08                              3
18   productcatalogservice-0        container.node-5.container_file_descriptors  1.027500e+01  1.000000e+01             30              both  2022-03-21T21:30:00+08:00         1.100000e+01         1.000000e+01                              3
19         paymentservice2-0                 container.node-6.container_sockets  4.000000e+00  3.450000e+00             30              both  2022-03-21T21:30:00+08:00         4.000000e+00         3.000000e+00                              2```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse loaded dataframes if available, otherwise load from CSVs
tz = pytz.timezone('Asia/Shanghai')
try:
    df_error
except NameError:
    df_error = pd.read_csv('error_logs.csv')
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# Prepare timezone-aware datetime columns and minute-floor timestamp for both frames
for df in (df_error, df_log):
    if 'ts_dt' not in df.columns:
        df['ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # floor to minute for grouping (ensure inclusive minute-bucket)
    df['ts_min'] = df['ts_dt'].dt.floor('T')

# Define incident window (inclusive)
start = tz.localize(pd.Timestamp('2022-03-21 21:30:00'))
end = tz.localize(pd.Timestamp('2022-03-21 21:59:00'))

# Filter to window
err_win = df_error[(df_error['ts_min'] >= start) & (df_error['ts_min'] <= end)].copy()
log_win = df_log[(df_log['ts_min'] >= start) & (df_log['ts_min'] <= end)].copy()

# 1) From error_logs.csv: group by timestamp (minute) and cmdb_id, count messages.
err_group = (
    err_win
    .groupby(['ts_min', 'cmdb_id'], sort=False)
    .size()
    .reset_index(name='count')
)

# Format timestamp column as ISO-like string (UTC+8)
err_group['timestamp_min'] = err_group['ts_min'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
err_group = err_group[['timestamp_min', 'cmdb_id', 'count']]

# Top 20 rows sorted by count desc
err_top20 = err_group.sort_values(by='count', ascending=False).head(20).reset_index(drop=True)

# 2) Total error_logs count per minute (summed across cmdb_id) for the window (only minutes where count>0)
err_total_per_min = (
    err_win
    .groupby('ts_min', sort=True)
    .size()
    .reset_index(name='count')
)
err_total_per_min['timestamp_min'] = err_total_per_min['ts_min'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
err_total_per_min = err_total_per_min[['timestamp_min', 'count']].sort_values(by='timestamp_min', ascending=True).reset_index(drop=True)

# 3) From log.csv: filter to log_name in {"log.error_count","log.row_count"} and sum per minute & log_name across cmdb_ids
log_filtered = log_win[log_win['log_name'].isin({"log.error_count", "log.row_count"})].copy()
log_summed = (
    log_filtered
    .groupby(['ts_min', 'log_name'], sort=True)['value']
    .sum()
    .reset_index()
)
log_summed['timestamp_min'] = log_summed['ts_min'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
log_summed = log_summed[['timestamp_min', 'log_name', 'value']].sort_values(by=['timestamp_min', 'log_name'], ascending=True).reset_index(drop=True)
log_summed.rename(columns={'value': 'summed_value'}, inplace=True)

# To honor the "Limit outputs to at most 60 rows in total", display:
# - err_top20 (up to 20)
# - err_total_per_min (all minutes where count>0) -- typically up to 30 rows
# - log_summed (first N rows such that total displayed rows <= 60)
rows_allowed = 60
rows_used = len(err_top20) + len(err_total_per_min)
remaining = max(0, rows_allowed - rows_used)
log_summed_display = log_summed.head(remaining)

# Final compact outputs (variables to display)
err_top20, err_total_per_min, log_summed_display

```
Out[4]:
```
```python
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse previously computed tables if available, otherwise recompute
try:
    err_top20, err_total_per_min, log_summed_display
except NameError:
    # load and compute minimal aggregates
    df_error = pd.read_csv('error_logs.csv')
    df_log = pd.read_csv('log.csv')
    for df in (df_error, df_log):
        df['ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        df['ts_min'] = df['ts_dt'].dt.floor('T')
    start = tz.localize(pd.Timestamp('2022-03-21 21:30:00'))
    end = tz.localize(pd.Timestamp('2022-03-21 21:59:00'))
    err_win = df_error[(df_error['ts_min'] >= start) & (df_error['ts_min'] <= end)].copy()
    log_win = df_log[(df_log['ts_min'] >= start) & (df_log['ts_min'] <= end)].copy()
    err_group = err_win.groupby(['ts_min', 'cmdb_id']).size().reset_index(name='count')
    err_group['timestamp_min'] = err_group['ts_min'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    err_top20 = err_group.sort_values('count', ascending=False).head(20)[['timestamp_min','cmdb_id','count']].reset_index(drop=True)
    err_total_per_min = err_win.groupby('ts_min').size().reset_index(name='count')
    err_total_per_min['timestamp_min'] = err_total_per_min['ts_min'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    err_total_per_min = err_total_per_min[['timestamp_min','count']].sort_values('timestamp_min').reset_index(drop=True)
    log_filtered = log_win[log_win['log_name'].isin({"log.error_count", "log.row_count"})].copy()
    log_summed = log_filtered.groupby(['ts_min','log_name'], sort=True)['value'].sum().reset_index()
    log_summed['timestamp_min'] = log_summed['ts_min'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    log_summed = log_summed[['timestamp_min','log_name','value']].rename(columns={'value':'summed_value'}).sort_values(['timestamp_min','log_name']).reset_index(drop=True)
    log_summed_display = log_summed.copy()

# Compute summary facts
total_error_logs = int(err_total_per_min['count'].sum()) if len(err_total_per_min)>0 else 0
peak_row = err_total_per_min.loc[err_total_per_min['count'].idxmax()] if len(err_total_per_min)>0 else None
peak_minute = peak_row['timestamp_min'] if peak_row is not None else ''
peak_count = int(peak_row['count']) if peak_row is not None else 0
minute_counts = err_total_per_min['count'].tolist()
min_minute_count = int(minute_counts) if isinstance(minute_counts, (int,)) else (int(min(minute_counts)) if minute_counts else 0)
max_minute_count = int(max(minute_counts)) if minute_counts else 0

# Top cmdb aggregated over window (compact)
# Recompute safe aggregation from err_top20 if full list not available
try:
    # If original err_win available, use it; else aggregate from err_group
    err_win
except NameError:
    # build from err_top20 (top 20 only) - fallback
    cmdb_counts = err_top20.groupby('cmdb_id')['count'].sum().sort_values(ascending=False)
else:
    cmdb_counts = (err_win.groupby('cmdb_id').size().sort_values(ascending=False))

top_cmdbs = ", ".join([f"{c}({int(cmdb_counts[c])})" for c in list(cmdb_counts.index[:6])])

# Check alignment between error_logs counts and log.error_count sums
# Build per-minute error_count from log_summed_display
log_err = log_summed_display[log_summed_display['log_name']=='log.error_count'][['timestamp_min','summed_value']].reset_index(drop=True)
# Merge to compare with err_total_per_min
cmp = pd.merge(err_total_per_min, log_err, on='timestamp_min', how='left')
all_equal = cmp.apply(lambda r: r['count']==r['summed_value'], axis=1).all() if len(cmp)>0 else False

# Compose plain-English summary string
summary_lines = [
    f"Error logs (window 2022-03-21 21:30 to 21:59 UTC+8):",
    f"- Total error log messages in window: {total_error_logs}.",
    f"- Per-minute peak: {peak_count} errors at {peak_minute}.",
    f"- Minute-level counts range: {min(minute_counts) if minute_counts else 0} to {max_minute_count} (shown for minutes with >0 counts).",
    f"- Top cmdb_ids by error message volume (example top 6): {top_cmdbs}.",
    "",
    f"Top grouped error rows (timestamp_min, cmdb_id, count) — top 20 by count (example rows):",
    # include a compact CSV-like snippet from err_top20 (up to 6 rows shown inline)
    "  " + "; ".join([f"{row['timestamp_min']},{row['cmdb_id']},{row['count']}" for _, row in err_top20.head(6).iterrows()]),
    "",
    "Aggregated error counts per minute (all minutes with >0 errors):",
    "  (first 6 minutes shown) " + "; ".join([f"{r['timestamp_min']}:{r['count']}" for _, r in err_total_per_min.head(6).iterrows()]),
    "",
    "Log metrics (window) — summed across cmdb_ids per minute for log.error_count and log.row_count:",
    "  (first 6 rows shown) " + "; ".join([f"{r['timestamp_min']},{r['log_name']},{r['summed_value']}" for _, r in log_summed_display.head(6).iterrows()]),
    "",
    f"Note: per-minute sums of log.error_count {'match' if all_equal else 'do NOT match'} the grouped error_logs counts (when compared minute-by-minute).",
    "",
    "Interpretation:",
    "- Most error messages come from frontend pods (frontend-2 and frontend-1 dominate top counts).",
    "- Errors are sustained across the 30-minute window with peaks around the minutes shown above.",
    "- log.error_count aggregated from log.csv aligns with error_logs counts, confirming the error volume.",
    "",
    "Recommendation: investigate frontend-2 and frontend-1 around 2022-03-21T21:30+08:00 and the peak minute(s) (e.g., the minute listed as peak) for root-cause traces and logs."
]

summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               timestamp_min     cmdb_id  count
0   2022-03-21T21:45:00+0800  frontend-2     17
1   2022-03-21T21:32:00+0800  frontend-2     16
2   2022-03-21T21:31:00+0800  frontend-2     16
3   2022-03-21T21:38:00+0800  frontend-2     15
4   2022-03-21T21:40:00+0800  frontend-2     15
5   2022-03-21T21:34:00+0800  frontend-2     15
6   2022-03-21T21:42:00+0800  frontend-2     15
7   2022-03-21T21:30:00+0800  frontend-2     14
8   2022-03-21T21:33:00+0800  frontend-2     13
9   2022-03-21T21:36:00+0800  frontend-2     13
10  2022-03-21T21:43:00+0800  frontend-2     13
11  2022-03-21T21:47:00+0800  frontend-2     13
12  2022-03-21T21:46:00+0800  frontend-2     12
13  2022-03-21T21:37:00+0800  frontend-1     12
14  2022-03-21T21:35:00+0800  frontend-1     12
15  2022-03-21T21:30:00+0800  frontend-1     12
16  2022-03-21T21:32:00+0800  frontend-1     12
17  2022-03-21T21:49:00+0800  frontend-2     12
18  2022-03-21T21:39:00+0800  frontend-1     12
19  2022-03-21T21:44:00+0800  frontend-2     12,                timestamp_min  count
0   2022-03-21T21:30:00+0800     34
1   2022-03-21T21:31:00+0800     33
2   2022-03-21T21:32:00+0800     35
3   2022-03-21T21:33:00+0800     29
4   2022-03-21T21:34:00+0800     35
..                       ...    ...
25  2022-03-21T21:55:00+0800     20
26  2022-03-21T21:56:00+0800     15
27  2022-03-21T21:57:00+0800     18
28  2022-03-21T21:58:00+0800     16
29  2022-03-21T21:59:00+0800     16

[30 rows x 2 columns],               timestamp_min         log_name  summed_value
0  2022-03-21T21:30:00+0800  log.error_count          34.0
1  2022-03-21T21:30:00+0800    log.row_count        9384.0
2  2022-03-21T21:31:00+0800  log.error_count          33.0
3  2022-03-21T21:31:00+0800    log.row_count        9158.0
4  2022-03-21T21:32:00+0800  log.error_count          35.0
5  2022-03-21T21:32:00+0800    log.row_count        9127.0
6  2022-03-21T21:33:00+0800  log.error_count          29.0
7  2022-03-21T21:33:00+0800    log.row_count        8805.0
8  2022-03-21T21:34:00+0800  log.error_count          35.0
9  2022-03-21T21:34:00+0800    log.row_count        9026.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_trace if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure timezone-aware datetime and minute floor exist
if 'ts_dt' not in df_trace.columns:
    df_trace['ts_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    if df_trace['ts_dt'].dt.tz is None:
        df_trace['ts_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

df_trace['ts_min'] = df_trace['ts_dt'].dt.floor('T')

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using the FULL series
thresholds_trace = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], sort=False)['value']
    .agg(p95=lambda x: float(x.quantile(0.95)), p5=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# 2) Filter to incident window (inclusive) 2022-03-21 21:30:00+08:00 through 2022-03-21 21:59:00+08:00
start = tz.localize(pd.Timestamp('2022-03-21 21:30:00'))
end = tz.localize(pd.Timestamp('2022-03-21 21:59:00'))
df_trace_win = df_trace[(df_trace['ts_dt'] >= start) & (df_trace['ts_dt'] <= end)].copy()

# Merge thresholds into windowed rows
df_t = df_trace_win.merge(thresholds_trace, on=['cmdb_id', 'trace_name'], how='left')

# Flag anomalies using global thresholds
df_t['high'] = df_t['value'] >= df_t['p95']
df_t['low'] = df_t['value'] <= df_t['p5']
df_t['is_anom'] = df_t['high'] | df_t['low']

# Limit to trace_name values referencing frontend origins
pattern = r'from_frontend-0|from_frontend-1|from_frontend-2|from_frontend2-0'
df_t_front = df_t[df_t['trace_name'].str.contains(pattern, regex=True, na=False)].copy()

# Aggregate per (cmdb_id, trace_name)
def summarize_trace_group(g):
    anom_count = int(g['is_anom'].sum())
    earliest = g.loc[g['is_anom'], 'ts_dt'].min() if anom_count > 0 else pd.NaT
    any_high = bool(g['high'].any())
    any_low = bool(g['low'].any())
    direction = ('both' if (any_high and any_low) else ('high' if any_high else ('low' if any_low else 'none')))
    return pd.Series({
        'p95': float(g['p95'].iloc[0]),
        'p5': float(g['p5'].iloc[0]),
        'anomaly_count': anom_count,
        'anomaly_direction': direction,
        'earliest_anomaly_timestamp': earliest,
        'max_value_in_window': float(g['value'].max()) if len(g) > 0 else np.nan,
        'min_value_in_window': float(g['value'].min()) if len(g) > 0 else np.nan
    })

summaries_trace = df_t_front.groupby(['cmdb_id', 'trace_name'], sort=False).apply(summarize_trace_group).reset_index()

# Keep only groups with anomaly_count > 0
summaries_trace = summaries_trace[summaries_trace['anomaly_count'] > 0].copy()

# Format earliest timestamp and round numeric columns for compactness
summaries_trace['earliest_anomaly_timestamp'] = summaries_trace['earliest_anomaly_timestamp'].apply(
    lambda x: x.isoformat() if pd.notnull(x) else ''
)
for c in ['p95', 'p5', 'max_value_in_window', 'min_value_in_window']:
    summaries_trace[c] = summaries_trace[c].round(6)

# Return top 20 by anomaly_count descending with requested columns
out_cols = ['cmdb_id', 'trace_name', 'p95', 'p5', 'anomaly_count', 'anomaly_direction',
            'earliest_anomaly_timestamp', 'max_value_in_window', 'min_value_in_window']

top20_frontend_traces = summaries_trace.sort_values(by='anomaly_count', ascending=False).head(20)[out_cols].reset_index(drop=True)

top20_frontend_traces

```
Out[5]:
```
```python
summary = (
    "Summary of trace anomalies (frontend-origin traces) for 2022-03-21 21:30–21:59 UTC+8:\n"
    "- Global P95/P5 were computed on the full series, then anomalies counted inside the 30-min window.\n"
    "- Top 20 results (filtered to trace_name from_frontend-*) all have anomaly_count = 30 (i.e., one anomaly per minute for the whole window).\n"
    "- anomaly_direction is 'both' for the listed rows. Many trace.error_rate series are constant zeros (p95 == p5 == 0 and window values == 0), causing them to be flagged as both high and low against global thresholds.\n"
    "- A few traces show small non-zero timing values (e.g., duration_p95 or duration_mean) with slight variability (small max/min differences) but still produce anomalies every minute.\n"
    "- Earliest anomaly for all shown rows is 2022-03-21T21:30:00+08:00.\n"
    "- Affected services in the top list include adservice, recommendationservice, cartservice, checkoutservice, productcatalogservice, currencyservice, shippingservice, etc.\n"
    "Interpretation and next steps:\n"
    "- Many detections are driven by constant or quantized trace metrics (especially error_rate == 0), which create false-positive 'both' anomalies. Treat constant-zero error_rate series cautiously.\n"
    "- Prioritize traces with genuine variability in the window (non-equal p95/p5 and visible max>min) for troubleshooting (e.g., duration metrics that deviate), and inspect traces/logs from frontend→service hops around 21:30 UTC+8.\n"
    "- Investigate frontend-origin calls to the listed services (starting with those showing non-zero durations) for root-cause events or config/deployment changes at ~21:30."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                           trace_name       p95     p5  anomaly_count anomaly_direction earliest_anomaly_timestamp  max_value_in_window  min_value_in_window
0                adservice-0     trace.from_frontend-0.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000
1                adservice-1     trace.from_frontend-0.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000
2                adservice-0     trace.from_frontend-1.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000
3                adservice-0     trace.from_frontend-2.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000
4    recommendationservice-2     trace.from_frontend-1.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000
5              cartservice-2     trace.from_frontend-1.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000
6             cartservice2-0    trace.from_frontend2-0.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000
7          shippingservice-0     trace.from_frontend-1.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000
8             cartservice2-0  trace.from_frontend2-0.duration_p95  0.001000  0.001             30              both  2022-03-21T21:30:00+08:00             0.001000                0.001
9    recommendationservice-2     trace.from_frontend-2.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000
10  recommendationservice2-0    trace.from_frontend2-0.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000
11   recommendationservice-2     trace.from_frontend-0.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000
12               adservice-1     trace.from_frontend-1.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000
13             cartservice-0     trace.from_frontend-2.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000
14             cartservice-1  trace.from_frontend-0.duration_mean  0.000039  0.000             30              both  2022-03-21T21:30:00+08:00             0.000143                0.000
15              adservice2-0    trace.from_frontend2-0.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000
16             cartservice-0     trace.from_frontend-1.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000
17         currencyservice-1     trace.from_frontend-0.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000
18   productcatalogservice-2     trace.from_frontend-0.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000
19   productcatalogservice-1     trace.from_frontend-1.error_rate  0.000000  0.000             30              both  2022-03-21T21:30:00+08:00             0.000000                0.000```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing dataframes if present, otherwise load
try:
    df_error
except NameError:
    df_error = pd.read_csv('error_logs.csv')
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# Ensure timezone-aware datetime and minute floor
for df in (df_error, df_log):
    if 'ts_dt' not in df.columns:
        df['ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    if 'ts_min' not in df.columns:
        df['ts_min'] = df['ts_dt'].dt.floor('T')

# Define incident window (inclusive)
start = tz.localize(pd.Timestamp('2022-03-21 21:30:00'))
end = tz.localize(pd.Timestamp('2022-03-21 21:59:00'))

# Filter to window
err_win = df_error[(df_error['ts_min'] >= start) & (df_error['ts_min'] <= end)].copy()
log_win = df_log[(df_log['ts_min'] >= start) & (df_log['ts_min'] <= end)].copy()

# Aggregate total error count per minute
err_total_per_min = err_win.groupby('ts_min').size().reset_index(name='total_error_count')

# Identify top 2 minutes by total_error_count (descending). If tie, earlier minute first.
top_minutes = err_total_per_min.sort_values(['total_error_count', 'ts_min'], ascending=[False, True]).head(2).reset_index(drop=True)

# Prepare per-minute top cmdb_id counts and log.error_count sums
rows = []
for _, r in top_minutes.iterrows():
    minute = r['ts_min']
    total_count = int(r['total_error_count'])
    # Top 10 cmdb_id counts for that minute
    cmdb_counts = (
        err_win[err_win['ts_min'] == minute]
        .groupby('cmdb_id')
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
    )
    top_cmdbs_list = [f"{row['cmdb_id']}({int(row['count'])})" for _, row in cmdb_counts.head(10).iterrows()]
    top_cmdbs_str = ",".join(top_cmdbs_list)
    # Summed log.error_count from log.csv for that minute
    log_err_sum_row = (
        log_win[(log_win['ts_min'] == minute) & (log_win['log_name'] == 'log.error_count')]
        .groupby('ts_min')['value']
        .sum()
        .reset_index(name='log_error_count_sum')
    )
    log_err_sum = float(log_err_sum_row['log_error_count_sum'].iloc[0]) if not log_err_sum_row.empty else 0.0
    rows.append({
        'timestamp_min': minute.strftime('%Y-%m-%dT%H:%M:%S%z'),
        'total_error_count': total_count,
        'top_cmdbs': top_cmdbs_str,
        'log_error_count_sum': log_err_sum
    })

top2_minutes_df = pd.DataFrame(rows, columns=['timestamp_min','total_error_count','top_cmdbs','log_error_count_sum'])

top2_minutes_df

```
Out[6]:
```
```python
# Build a concise plain-English summary from top2_minutes_df
try:
    df = top2_minutes_df
except NameError:
    summary = "Top-2 minute results not available in memory."
else:
    r0 = df.iloc[0]
    r1 = df.iloc[1] if df.shape[0] > 1 else None
    lines = []
    lines.append("Top 2 minutes by error message count (window 2022-03-21 21:30–21:59 UTC+8):")
    lines.append(f"- {r0['timestamp_min']}: total_error_count={int(r0['total_error_count'])}; top_cmdbs={r0['top_cmdbs']}; log.error_count_sum={r0['log_error_count_sum']}")
    if r1 is not None:
        lines.append(f"- {r1['timestamp_min']}: total_error_count={int(r1['total_error_count'])}; top_cmdbs={r1['top_cmdbs']}; log.error_count_sum={r1['log_error_count_sum']}")
    lines.append("")
    lines.append("Summary interpretation:")
    lines.append("- Both peak minutes (21:32 and 21:34 UTC+8) have 35 error messages each.")
    lines.append("- frontend pods (especially frontend-2, then frontend-1 and frontend-0) are the main contributors in those minutes.")
    lines.append("- The summed log.error_count equals the error_logs counts for these minutes (35.0), confirming the error volume.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

timestamp_min  total_error_count                                    top_cmdbs  log_error_count_sum
0  2022-03-21T21:32:00+0800                 35  frontend-2(16),frontend-1(12),frontend-0(7)                 35.0
1  2022-03-21T21:34:00+0800                 35  frontend-2(15),frontend-1(12),frontend-0(8)                 35.0```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse df_metric if available, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure timezone and timestamp columns (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'ts_dt' not in df_metric.columns:
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware in Asia/Shanghai
    if df_metric['ts_dt'].dt.tz is None:
        df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df_metric['ts_dt'] = df_metric['ts_dt'].dt.tz_convert(tz)

# floor to minute for exact-minute extraction
df_metric['ts_min'] = df_metric['ts_dt'].dt.floor('T')

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) on the FULL series (rule 9)
thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], sort=False)['value']
    .agg(p95=lambda x: float(x.quantile(0.95)), p5=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# Exclude constant/config metrics where p95 == p5 (will be excluded from results)
thresholds_nonconst = thresholds[thresholds['p95'] != thresholds['p5']].copy()

# Target minutes (UTC+8)
minutes = [
    tz.localize(pd.Timestamp('2022-03-21 21:32:00')),
    tz.localize(pd.Timestamp('2022-03-21 21:34:00'))
]

results = {}
for minute in minutes:
    # Filter metric rows at this exact minute
    df_min = df_metric[df_metric['ts_min'] == minute].copy()
    if df_min.empty:
        # empty placeholder
        results[minute] = pd.DataFrame(columns=[
            'minute','cmdb_id','kpi_name','p95','p5','value_at_minute','anomaly_direction','abs_diff'
        ])
        continue
    # Merge with global thresholds (non-constant only)
    df_min = df_min.merge(thresholds_nonconst, on=['cmdb_id','kpi_name'], how='inner')
    if df_min.empty:
        results[minute] = pd.DataFrame(columns=[
            'minute','cmdb_id','kpi_name','p95','p5','value_at_minute','anomaly_direction','abs_diff'
        ])
        continue
    # Determine breaches at that minute
    df_min['high'] = df_min['value'] >= df_min['p95']
    df_min['low'] = df_min['value'] <= df_min['p5']
    df_breach = df_min[(df_min['high']) | (df_min['low'])].copy()
    if df_breach.empty:
        results[minute] = pd.DataFrame(columns=[
            'minute','cmdb_id','kpi_name','p95','p5','value_at_minute','anomaly_direction','abs_diff'
        ])
        continue
    # Set direction and abs_diff
    def compute_direction_abs(row):
        if row['high']:
            return 'high', abs(float(row['value']) - float(row['p95']))
        else:
            return 'low', abs(float(row['value']) - float(row['p5']))
    dir_abs = df_breach.apply(compute_direction_abs, axis=1, result_type='expand')
    df_breach['anomaly_direction'] = dir_abs[0]
    df_breach['abs_diff'] = dir_abs[1]
    # Prepare output columns
    df_out = df_breach[['cmdb_id','kpi_name','p95','p5','value','anomaly_direction','abs_diff']].copy()
    df_out.rename(columns={'value':'value_at_minute'}, inplace=True)
    # Round numeric columns for compactness
    for c in ['p95','p5','value_at_minute','abs_diff']:
        df_out[c] = df_out[c].astype(float).round(6)
    # Add minute column as ISO string with timezone
    df_out.insert(0, 'minute', minute.isoformat())
    # Sort by abs_diff desc and limit to top 20
    df_out = df_out.sort_values(by='abs_diff', ascending=False).head(20).reset_index(drop=True)
    results[minute] = df_out

# Final compact outputs: two DataFrames (up to 20 rows each)
minute_2132, minute_2134 = results[minutes[0]], results[minutes[1]]

minute_2132, minute_2134

```
Out[7]:
```
```python
# Build a concise plain-English summary using the previously computed DataFrames minute_2132 and minute_2134.
try:
    df1 = minute_2132
    df2 = minute_2134
except NameError:
    summary = "Result DataFrames not found in memory. Please re-run the metric anomaly extraction step."
else:
    def top_examples(df, n=3):
        rows = []
        for _, r in df.head(n).iterrows():
            cm = r['cmdb_id']
            kpi = r['kpi_name']
            dirc = r['anomaly_direction']
            diff = float(r['abs_diff'])
            # format diff with appropriate scale
            if diff >= 1e6:
                diff_s = f"{diff:,.0f}"
            elif diff >= 1e3:
                diff_s = f"{diff:,.1f}"
            else:
                diff_s = f"{diff:.3f}"
            rows.append(f"{cm}({kpi}, {dirc}, Δ={diff_s})")
        return "; ".join(rows) if rows else "none"
    s_lines = []
    for df in (df1, df2):
        if df.empty:
            s_lines.append(f"{df['minute'].iloc[0] if 'minute' in df.columns else 'N/A'}: no breaches (after excluding constant/config metrics).")
            continue
        minute = df['minute'].iloc[0]
        total_breaches = int(df.shape[0])
        # top example list
        examples = top_examples(df, n=5)
        # dominant direction counts
        dir_counts = df['anomaly_direction'].value_counts().to_dict()
        dirs = ", ".join([f"{k}:{v}" for k, v in dir_counts.items()])
        s_lines.append(f"{minute}: {total_breaches} threshold breaches; directions: {dirs}; top examples: {examples}.")
    # Short interpretation
    s_lines.append("Interpretation: At 21:32 most significant breaches are node and container memory/fs metrics (e.g., frontend-2 memory_cache high, node fs inodes/disk changes and several container RSS highs). At 21:34 prominent breaches include service container memory_rss spikes (currencyservice, frontend) and several mesh/latency-like mesh.* metrics. These are non-constant, measurable deviations (we excluded constant/config metrics).")
    s_lines.append("Recommendation: Focus on the listed high memory and filesystem counters for the respective minutes (21:32 and 21:34 UTC+8) when investigating root cause.")
    summary = "\n".join(s_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                       minute                  cmdb_id                                          kpi_name           p95            p5  value_at_minute anomaly_direction      abs_diff
0   2022-03-21T21:32:00+08:00               frontend-2           container.node-5.container_memory_cache  2.140492e+08  2.243379e+07     2.160108e+08              high  1.961574e+06
1   2022-03-21T21:32:00+08:00               adservice2  runtime.java_lang_Threading_CurrentThreadCpuTime  1.671253e+10  1.618499e+10     1.618375e+10               low  1.245751e+06
2   2022-03-21T21:32:00+08:00               adservice2     runtime.java_lang_Memory_HeapMemoryUsage_used  5.568227e+07  5.223281e+07     5.172646e+07               low  5.063521e+05
3   2022-03-21T21:32:00+08:00                   node-4                            system.fs.inodes.total  6.449805e+09  6.445531e+09     6.450142e+09              high  3.366912e+05
4   2022-03-21T21:32:00+08:00                   node-4                             system.fs.inodes.free  6.447210e+09  6.442937e+09     6.447547e+09              high  3.366912e+05
5   2022-03-21T21:32:00+08:00                   node-2                             system.fs.inodes.free  4.845168e+09  4.842079e+09     4.845387e+09              high  2.186496e+05
6   2022-03-21T21:32:00+08:00                   node-1                             system.fs.inodes.free  4.845198e+09  4.842104e+09     4.845416e+09              high  2.186496e+05
7   2022-03-21T21:32:00+08:00                   node-1                            system.fs.inodes.total  4.847143e+09  4.844050e+09     4.847362e+09              high  2.186496e+05
8   2022-03-21T21:32:00+08:00                   node-2                            system.fs.inodes.total  4.847143e+09  4.844054e+09     4.847362e+09              high  2.186496e+05
9   2022-03-21T21:32:00+08:00  productcatalogservice-0             container.node-5.container_memory_rss  1.522790e+07  1.482332e+07     1.538867e+07              high  1.607680e+05
10  2022-03-21T21:32:00+08:00       checkoutservice2-0             container.node-5.container_memory_rss  1.572413e+07  1.491343e+07     1.585357e+07              high  1.294336e+05
11  2022-03-21T21:32:00+08:00  productcatalogservice-1             container.node-5.container_memory_rss  1.649992e+07  1.612216e+07     1.662771e+07              high  1.277952e+05
12  2022-03-21T21:32:00+08:00                   node-6                            system.fs.inodes.total  4.312889e+09  4.310107e+09     4.312977e+09              high  8.765440e+04
13  2022-03-21T21:32:00+08:00                   node-6                             system.fs.inodes.free  4.310974e+09  4.308192e+09     4.311062e+09              high  8.765440e+04
14  2022-03-21T21:32:00+08:00                   node-4                                  system.disk.used  5.808544e+09  5.807476e+09     5.807391e+09               low  8.417280e+04
15  2022-03-21T21:32:00+08:00                   node-4                                  system.disk.free  1.800371e+09  1.799302e+09     1.800456e+09              high  8.417280e+04
16  2022-03-21T21:32:00+08:00                   node-3                            system.fs.inodes.total  4.847116e+09  4.843838e+09     4.847187e+09              high  7.119360e+04
17  2022-03-21T21:32:00+08:00                   node-3                             system.fs.inodes.free  4.845162e+09  4.841885e+09     4.845234e+09              high  7.119360e+04
18  2022-03-21T21:32:00+08:00                   node-1                                  system.disk.used  4.361335e+09  4.360551e+09     4.360496e+09               low  5.529600e+04
19  2022-03-21T21:32:00+08:00                   node-1                                  system.disk.free  1.493778e+09  1.492995e+09     1.493834e+09              high  5.512320e+04,                        minute                  cmdb_id                                           kpi_name           p95            p5  value_at_minute anomaly_direction       abs_diff
0   2022-03-21T21:34:00+08:00        currencyservice-1              container.node-5.container_memory_rss  3.546388e+07  3.488164e+07       35688448.0              high  224563.200000
1   2022-03-21T21:34:00+08:00               frontend-0              container.node-5.container_memory_rss  1.918403e+07  1.868831e+07       19322880.0              high  138854.400000
2   2022-03-21T21:34:00+08:00       currencyservice2-0              container.node-6.container_memory_rss  3.242793e+07  3.148646e+07       32546816.0              high  118886.400000
3   2022-03-21T21:34:00+08:00               adservice2  runtime.java_lang_MemoryPool_Usage_used.Eden_S...  1.116528e+07  7.211197e+06       11248800.0              high   83520.300000
4   2022-03-21T21:34:00+08:00         paymentservice-0              container.node-5.container_memory_rss  3.421143e+07  3.337421e+07       34238464.0              high   27033.600000
5   2022-03-21T21:34:00+08:00               frontend-1  mesh.source.frontend.productcatalogservice.ist...  3.977300e+05  2.513712e+05         407650.0              high    9920.000000
6   2022-03-21T21:34:00+08:00  productcatalogservice-2  mesh.destination.frontend.productcatalogservic...  3.928550e+05  1.835862e+05         401475.0              high    8620.000000
7   2022-03-21T21:34:00+08:00  productcatalogservice-2  mesh.destination.recommendationservice.product...  1.135062e+05  5.348750e+04         121000.0              high    7493.750000
8   2022-03-21T21:34:00+08:00       shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....  5.098625e+04  2.182125e+04          56675.0              high    5688.750000
9   2022-03-21T21:34:00+08:00  productcatalogservice-2  mesh.source.productcatalogservice.jaeger-colle...  3.885925e+05  2.141062e+05         393125.0              high    4532.500000
10  2022-03-21T21:34:00+08:00  productcatalogservice-2  mesh.destination.frontend.productcatalogservic...  1.739085e+05  8.097438e+04         177772.5              high    3864.000000
11  2022-03-21T21:34:00+08:00                adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  2.299040e+07  2.297395e+07       22970816.0               low    3137.066667
12  2022-03-21T21:34:00+08:00                adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  2.299040e+07  2.297395e+07       22970816.0               low    3137.066667
13  2022-03-21T21:34:00+08:00                adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  3.616449e+05  1.357288e+05         132718.0               low    3010.800000
14  2022-03-21T21:34:00+08:00                adservice  runtime.java_lang_MemoryPool_Usage_used.Code_C...  2.299040e+07  2.297325e+07       22970816.0               low    2430.400000
15  2022-03-21T21:34:00+08:00  productcatalogservice-2  mesh.source.productcatalogservice.basic-tidb.i...  2.052365e+05  9.636455e+04         207601.5              high    2364.975000
16  2022-03-21T21:34:00+08:00               frontend-1  mesh.source.frontend.recommendationservice.ist...  6.200000e+04  3.832375e+04          64325.0              high    2325.000000
17  2022-03-21T21:34:00+08:00                adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  3.654748e+05  1.329894e+05         130886.0               low    2103.433333
18  2022-03-21T21:34:00+08:00  productcatalogservice-2  mesh.destination.recommendationservice.product...  3.074988e+04  1.449025e+04          32780.0              high    2030.125000
19  2022-03-21T21:34:00+08:00                adservice   runtime.java_lang_Memory_NonHeapMemoryUsage_used  5.810060e+07  5.808304e+07       58081016.0               low    2027.733333)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing dataframes and timezone if available
tz = pytz.timezone('Asia/Shanghai')
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure datetime columns and minute floor in UTC+8
for df in (df_trace, df_metric):
    if 'ts_dt' not in df.columns:
        df['ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        if df['ts_dt'].dt.tz is None:
            df['ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        else:
            df['ts_dt'] = df['ts_dt'].dt.tz_convert(tz)
    df['ts_min'] = df['ts_dt'].dt.floor('T')

# Target minutes (UTC+8)
minutes = [
    tz.localize(pd.Timestamp('2022-03-21 21:32:00')),
    tz.localize(pd.Timestamp('2022-03-21 21:34:00'))
]

# --- Part 1: trace.csv (trace_name contains 'from_frontend') ---
# Compute global p95/p5 per (cmdb_id, trace_name)
thresholds_trace = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], sort=False)['value']
    .agg(p95=lambda x: float(x.quantile(0.95)), p5=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# Exclude constant/config where p95 == p5
thresholds_trace_nc = thresholds_trace[thresholds_trace['p95'] != thresholds_trace['p5']].copy()

trace_results = {}
pattern = 'from_frontend'  # matches from_frontend and from_frontend2-0 etc.
for minute in minutes:
    # rows at this minute and matching pattern
    df_tmin = df_trace[(df_trace['ts_min'] == minute) & (df_trace['trace_name'].str.contains(pattern, na=False))].copy()
    if df_tmin.empty:
        trace_results[minute] = pd.DataFrame(columns=[
            'minute','cmdb_id','trace_name','p95','p5','value_at_minute','anomaly_direction','abs_diff'
        ])
        continue
    # merge with global thresholds (non-constant)
    df_tm = df_tmin.merge(thresholds_trace_nc, on=['cmdb_id','trace_name'], how='inner')
    if df_tm.empty:
        trace_results[minute] = pd.DataFrame(columns=[
            'minute','cmdb_id','trace_name','p95','p5','value_at_minute','anomaly_direction','abs_diff'
        ])
        continue
    # compute breach flags
    df_tm['high'] = df_tm['value'] >= df_tm['p95']
    df_tm['low'] = df_tm['value'] <= df_tm['p5']
    df_b = df_tm[(df_tm['high']) | (df_tm['low'])].copy()
    if df_b.empty:
        trace_results[minute] = pd.DataFrame(columns=[
            'minute','cmdb_id','trace_name','p95','p5','value_at_minute','anomaly_direction','abs_diff'
        ])
        continue
    # direction and abs_diff
    df_b['anomaly_direction'] = np.where(df_b['high'], 'high', 'low')
    df_b['abs_diff'] = np.where(df_b['high'],
                                (df_b['value'] - df_b['p95']).abs(),
                                (df_b['value'] - df_b['p5']).abs())
    df_out = df_b[['cmdb_id','trace_name','p95','p5','value','anomaly_direction','abs_diff']].copy()
    df_out.rename(columns={'value':'value_at_minute'}, inplace=True)
    # rounding
    for c in ['p95','p5','value_at_minute','abs_diff']:
        df_out[c] = df_out[c].astype(float).round(6)
    df_out.insert(0, 'minute', minute.isoformat())
    df_out = df_out.sort_values(by='abs_diff', ascending=False).head(10).reset_index(drop=True)
    trace_results[minute] = df_out

# --- Part 2: metric.csv ---
# Compute global p95/p5 per (cmdb_id, kpi_name)
thresholds_metric = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], sort=False)['value']
    .agg(p95=lambda x: float(x.quantile(0.95)), p5=lambda x: float(x.quantile(0.05)))
    .reset_index()
)
thresholds_metric_nc = thresholds_metric[thresholds_metric['p95'] != thresholds_metric['p5']].copy()

metric_results = {}
for minute in minutes:
    df_mmin = df_metric[df_metric['ts_min'] == minute].copy()
    if df_mmin.empty:
        metric_results[minute] = pd.DataFrame(columns=[
            'minute','cmdb_id','kpi_name','p95','p5','value_at_minute','anomaly_direction','abs_diff'
        ])
        continue
    df_mm = df_mmin.merge(thresholds_metric_nc, on=['cmdb_id','kpi_name'], how='inner')
    if df_mm.empty:
        metric_results[minute] = pd.DataFrame(columns=[
            'minute','cmdb_id','kpi_name','p95','p5','value_at_minute','anomaly_direction','abs_diff'
        ])
        continue
    df_mm['high'] = df_mm['value'] >= df_mm['p95']
    df_mm['low'] = df_mm['value'] <= df_mm['p5']
    df_b = df_mm[(df_mm['high']) | (df_mm['low'])].copy()
    if df_b.empty:
        metric_results[minute] = pd.DataFrame(columns=[
            'minute','cmdb_id','kpi_name','p95','p5','value_at_minute','anomaly_direction','abs_diff'
        ])
        continue
    df_b['anomaly_direction'] = np.where(df_b['high'], 'high', 'low')
    df_b['abs_diff'] = np.where(df_b['high'],
                                (df_b['value'] - df_b['p95']).abs(),
                                (df_b['value'] - df_b['p5']).abs())
    df_out = df_b[['cmdb_id','kpi_name','p95','p5','value','anomaly_direction','abs_diff']].copy()
    df_out.rename(columns={'value':'value_at_minute'}, inplace=True)
    for c in ['p95','p5','value_at_minute','abs_diff']:
        df_out[c] = df_out[c].astype(float).round(6)
    df_out.insert(0, 'minute', minute.isoformat())
    df_out = df_out.sort_values(by='abs_diff', ascending=False).head(10).reset_index(drop=True)
    metric_results[minute] = df_out

# Prepare final compact outputs (up to 40 rows total: 10 trace + 10 metric per minute)
trace_2132 = trace_results[minutes[0]]
trace_2134 = trace_results[minutes[1]]
metric_2132 = metric_results[minutes[0]]
metric_2134 = metric_results[minutes[1]]

# Display compact tables (as variables, not prints)
trace_2132, trace_2134, metric_2132, metric_2134

```
Out[8]:
```
summary = (
    "Summary for minutes 2022-03-21T21:32:00+08:00 and 2022-03-21T21:34:00+08:00 (thresholds = global P95/P5):\n\n"
    "21:32 (UTC+8):\n"
    "- Traces: top breaches are 'trace.from_frontend-2.row_count' on multiple services (productcatalogservice-1/2/0, cartservice-*, recommendationservice-*, adservice-1). These are 'high' vs P95 (example Δ≈11.9,11.35,10.9, smaller for others).\n"
    "- Metrics: top breaches include frontend-2.container_memory_cache (high, Δ≈1.96e6), several node filesystem inode and disk counters (node-1/2/3/4/6 fs.inodes.total/free high), and container RSS/memory highs on productcatalog and checkoutservice. Also some adservice2 runtime/memory values were low vs P5.\n\n"
    "21:34 (UTC+8):\n"
    "- Traces: top breaches are 'trace.from_frontend-1.row_count' on productcatalogservice instances (high, Δ≈9–8.8) and other frontend-origin row_count increases across recommendationservice and cartservice.\n"
    "- Metrics: notable breaches are container memory_rss spikes (currencyservice-1, frontend-0, currencyservice2-0, paymentservice-0) and several mesh-related metrics (mesh.source/destination.*) showing high deviations. Some adservice JVM/GC memory metrics show low-side breaches.\n\n"
    "Interpretation and guidance:\n"
    "- Many of the largest trace anomalies are elevated request row_counts from frontend-origin calls into productcatalogservice and related services at these minutes.\n"
    "- Metric anomalies point to container memory/RSS increases and node filesystem/inode changes at 21:32, and memory/RSS + mesh call-volume/latency-like metrics at 21:34.\n"
    "- Constants/config metrics were excluded (p95==p5). Focus investigation on the services and pods listed (productcatalogservice, frontend, currencyservice, adservice, nodes implicated) around 21:32–21:34 for root-cause (memory pressure, increased frontend traffic, or mesh call spikes)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                      minute                  cmdb_id                       trace_name     p95     p5  value_at_minute anomaly_direction  abs_diff
0  2022-03-21T21:32:00+08:00  productcatalogservice-1  trace.from_frontend-2.row_count  118.10  40.00            130.0              high     11.90
1  2022-03-21T21:32:00+08:00  productcatalogservice-2  trace.from_frontend-2.row_count  117.65  39.45            129.0              high     11.35
2  2022-03-21T21:32:00+08:00  productcatalogservice-0  trace.from_frontend-2.row_count  118.10  39.45            129.0              high     10.90
3  2022-03-21T21:32:00+08:00            cartservice-2  trace.from_frontend-2.row_count   22.55   7.45             25.0              high      2.45
4  2022-03-21T21:32:00+08:00  recommendationservice-1  trace.from_frontend-2.row_count   18.55   6.00             21.0              high      2.45
5  2022-03-21T21:32:00+08:00  recommendationservice-0  trace.from_frontend-2.row_count   18.00   6.00             20.0              high      2.00
6  2022-03-21T21:32:00+08:00            cartservice-1  trace.from_frontend-2.row_count   22.00   7.45             24.0              high      2.00
7  2022-03-21T21:32:00+08:00            cartservice-0  trace.from_frontend-2.row_count   22.55   7.45             24.0              high      1.45
8  2022-03-21T21:32:00+08:00  recommendationservice-2  trace.from_frontend-2.row_count   18.55   6.00             20.0              high      1.45
9  2022-03-21T21:32:00+08:00              adservice-1  trace.from_frontend-2.row_count   16.55   5.35             17.0              high      0.45,                       minute                  cmdb_id                       trace_name     p95     p5  value_at_minute anomaly_direction  abs_diff
0  2022-03-21T21:34:00+08:00  productcatalogservice-2  trace.from_frontend-1.row_count   91.20  50.15            101.0              high      9.80
1  2022-03-21T21:34:00+08:00  productcatalogservice-0  trace.from_frontend-1.row_count   91.65  50.60            101.0              high      9.35
2  2022-03-21T21:34:00+08:00  productcatalogservice-1  trace.from_frontend-1.row_count   92.20  50.15            101.0              high      8.80
3  2022-03-21T21:34:00+08:00  recommendationservice-0  trace.from_frontend-1.row_count   14.00   7.45             16.0              high      2.00
4  2022-03-21T21:34:00+08:00  recommendationservice-2  trace.from_frontend-1.row_count   14.55   7.45             16.0              high      1.45
5  2022-03-21T21:34:00+08:00  recommendationservice-1  trace.from_frontend-1.row_count   14.55   7.45             16.0              high      1.45
6  2022-03-21T21:34:00+08:00  productcatalogservice-2  trace.from_frontend-2.row_count  117.65  39.45            119.0              high      1.35
7  2022-03-21T21:34:00+08:00            cartservice-1  trace.from_frontend-1.row_count   17.00  10.00             18.0              high      1.00
8  2022-03-21T21:34:00+08:00            cartservice-0  trace.from_frontend-1.row_count   17.00  10.00             18.0              high      1.00
9  2022-03-21T21:34:00+08:00            cartservice-2  trace.from_frontend-1.row_count   17.00   9.45             18.0              high      1.00,                       minute                  cmdb_id                                          kpi_name           p95            p5  value_at_minute anomaly_direction      abs_diff
0  2022-03-21T21:32:00+08:00               frontend-2           container.node-5.container_memory_cache  2.140492e+08  2.243379e+07     2.160108e+08              high  1.961574e+06
1  2022-03-21T21:32:00+08:00               adservice2  runtime.java_lang_Threading_CurrentThreadCpuTime  1.671253e+10  1.618499e+10     1.618375e+10               low  1.245751e+06
2  2022-03-21T21:32:00+08:00               adservice2     runtime.java_lang_Memory_HeapMemoryUsage_used  5.568227e+07  5.223281e+07     5.172646e+07               low  5.063521e+05
3  2022-03-21T21:32:00+08:00                   node-4                            system.fs.inodes.total  6.449805e+09  6.445531e+09     6.450142e+09              high  3.366912e+05
4  2022-03-21T21:32:00+08:00                   node-4                             system.fs.inodes.free  6.447210e+09  6.442937e+09     6.447547e+09              high  3.366912e+05
5  2022-03-21T21:32:00+08:00                   node-2                             system.fs.inodes.free  4.845168e+09  4.842079e+09     4.845387e+09              high  2.186496e+05
6  2022-03-21T21:32:00+08:00                   node-1                             system.fs.inodes.free  4.845198e+09  4.842104e+09     4.845416e+09              high  2.186496e+05
7  2022-03-21T21:32:00+08:00                   node-1                            system.fs.inodes.total  4.847143e+09  4.844050e+09     4.847362e+09              high  2.186496e+05
8  2022-03-21T21:32:00+08:00                   node-2                            system.fs.inodes.total  4.847143e+09  4.844054e+09     4.847362e+09              high  2.186496e+05
9  2022-03-21T21:32:00+08:00  productcatalogservice-0             container.node-5.container_memory_rss  1.522790e+07  1.482332e+07     1.538867e+07              high  1.607680e+05,                       minute                  cmdb_id                                           kpi_name          p95            p5  value_at_minute anomaly_direction   abs_diff
0  2022-03-21T21:34:00+08:00        currencyservice-1              container.node-5.container_memory_rss  35463884.80  3.488164e+07       35688448.0              high  224563.20
1  2022-03-21T21:34:00+08:00               frontend-0              container.node-5.container_memory_rss  19184025.60  1.868831e+07       19322880.0              high  138854.40
2  2022-03-21T21:34:00+08:00       currencyservice2-0              container.node-6.container_memory_rss  32427929.60  3.148646e+07       32546816.0              high  118886.40
3  2022-03-21T21:34:00+08:00               adservice2  runtime.java_lang_MemoryPool_Usage_used.Eden_S...  11165279.70  7.211197e+06       11248800.0              high   83520.30
4  2022-03-21T21:34:00+08:00         paymentservice-0              container.node-5.container_memory_rss  34211430.40  3.337421e+07       34238464.0              high   27033.60
5  2022-03-21T21:34:00+08:00               frontend-1  mesh.source.frontend.productcatalogservice.ist...    397730.00  2.513712e+05         407650.0              high    9920.00
6  2022-03-21T21:34:00+08:00  productcatalogservice-2  mesh.destination.frontend.productcatalogservic...    392855.00  1.835862e+05         401475.0              high    8620.00
7  2022-03-21T21:34:00+08:00  productcatalogservice-2  mesh.destination.recommendationservice.product...    113506.25  5.348750e+04         121000.0              high    7493.75
8  2022-03-21T21:34:00+08:00       shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     50986.25  2.182125e+04          56675.0              high    5688.75
9  2022-03-21T21:34:00+08:00  productcatalogservice-2  mesh.source.productcatalogservice.jaeger-colle...    388592.50  2.141062e+05         393125.0              high    4532.50)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure required columns exist
required_cols = {'cmdb_id', 'kpi_name'}
if not required_cols.issubset(set(df_metric.columns)):
    raise ValueError("metric.csv missing required columns")

# Target cmdb_ids
targets = [
    'frontend-2',
    'productcatalogservice-0',
    'productcatalogservice-1',
    'productcatalogservice-2',
    'currencyservice-1',
    'frontend-0'
]

# Group counts per (cmdb_id, kpi_name)
grp = (
    df_metric
    .loc[df_metric['cmdb_id'].isin(targets), ['cmdb_id', 'kpi_name']]
    .groupby(['cmdb_id', 'kpi_name'], sort=False)
    .size()
    .reset_index(name='count')
)

# Prepare per-cmdb compact tables (limit to top 50 kpi_name by count)
frontend2_kpis = grp[grp['cmdb_id']=='frontend-2'].sort_values('count', ascending=False).head(50).reset_index(drop=True)
productcatalogservice_0_kpis = grp[grp['cmdb_id']=='productcatalogservice-0'].sort_values('count', ascending=False).head(50).reset_index(drop=True)
productcatalogservice_1_kpis = grp[grp['cmdb_id']=='productcatalogservice-1'].sort_values('count', ascending=False).head(50).reset_index(drop=True)
productcatalogservice_2_kpis = grp[grp['cmdb_id']=='productcatalogservice-2'].sort_values('count', ascending=False).head(50).reset_index(drop=True)
currencyservice_1_kpis = grp[grp['cmdb_id']=='currencyservice-1'].sort_values('count', ascending=False).head(50).reset_index(drop=True)
frontend0_kpis = grp[grp['cmdb_id']=='frontend-0'].sort_values('count', ascending=False).head(50).reset_index(drop=True)

# Also provide a one-line summary counts per cmdb (compact)
summary_counts = grp.groupby('cmdb_id', sort=False)['kpi_name'].nunique().reset_index().rename(columns={'kpi_name':'distinct_kpi_names'})

# Display compact tables (up to 50 rows each). Variables returned for the Controller to read:
frontend2_kpis, productcatalogservice_0_kpis, productcatalogservice_1_kpis, productcatalogservice_2_kpis, currencyservice_1_kpis, frontend0_kpis, summary_counts

```
Out[9]:
```
```python
# Build a concise plain-English summary of distinct kpi_name lists per requested cmdb_id.
try:
    frontend2_kpis, productcatalogservice_0_kpis, productcatalogservice_1_kpis, productcatalogservice_2_kpis, currencyservice_1_kpis, frontend0_kpis, summary_counts
except NameError:
    # fallback: load minimal info from metric.csv
    import pandas as pd, pytz
    tz = pytz.timezone('Asia/Shanghai')
    df_metric = pd.read_csv('metric.csv')
    grp = (df_metric[df_metric['cmdb_id'].isin([
            'frontend-2','productcatalogservice-0','productcatalogservice-1','productcatalogservice-2','currencyservice-1','frontend-0'
        ])]
        .groupby(['cmdb_id','kpi_name'], sort=False).size().reset_index(name='count'))
    frontend2_kpis = grp[grp['cmdb_id']=='frontend-2'].sort_values('count', ascending=False).head(50).reset_index(drop=True)
    productcatalogservice_0_kpis = grp[grp['cmdb_id']=='productcatalogservice-0'].sort_values('count', ascending=False).head(50).reset_index(drop=True)
    productcatalogservice_1_kpis = grp[grp['cmdb_id']=='productcatalogservice-1'].sort_values('count', ascending=False).head(50).reset_index(drop=True)
    productcatalogservice_2_kpis = grp[grp['cmdb_id']=='productcatalogservice-2'].sort_values('count', ascending=False).head(50).reset_index(drop=True)
    currencyservice_1_kpis = grp[grp['cmdb_id']=='currencyservice-1'].sort_values('count', ascending=False).head(50).reset_index(drop=True)
    frontend0_kpis = grp[grp['cmdb_id']=='frontend-0'].sort_values('count', ascending=False).head(50).reset_index(drop=True)
    summary_counts = grp.groupby('cmdb_id')['kpi_name'].nunique().reset_index().rename(columns={'kpi_name':'distinct_kpi_names'})

def top_examples(df, n=3):
    return ", ".join([f"{row['kpi_name']}({int(row['count'])})" for _, row in df.head(n).iterrows()]) or "none"

# Gather summary facts
counts_map = {r['cmdb_id']: int(r['distinct_kpi_names']) for _, r in summary_counts.iterrows()}

lines = []
lines.append("Distinct kpi_name summary for requested cmdb_ids (compact):")
for cmdb, df in [
    ('frontend-2', frontend2_kpis),
    ('productcatalogservice-0', productcatalogservice_0_kpis),
    ('productcatalogservice-1', productcatalogservice_1_kpis),
    ('productcatalogservice-2', productcatalogservice_2_kpis),
    ('currencyservice-1', currencyservice_1_kpis),
    ('frontend-0', frontend0_kpis),
]:
    distinct_n = counts_map.get(cmdb, df['kpi_name'].nunique() if not df.empty else 0)
    top3 = top_examples(df, n=3)
    lines.append(f"- {cmdb}: {distinct_n} distinct kpi_name(s); top examples: {top3} (counts show rows per kpi, many are 30).")

lines.append("")
lines.append("Notes:")
lines.append("- Results limited to up to 50 kpi_name entries per cmdb_id; counts represent number of rows observed for each kpi_name in metric.csv.")
lines.append("- Many container/node KPIs (container_*, system.*) appear as top entries and typically have count=30 in this dataset.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count
0   frontend-2         container.node-5.container_cpu_cfs_periods     30
1   frontend-2  container.node-5.container_cpu_cfs_throttled_p...     30
2   frontend-2  container.node-5.container_cpu_cfs_throttled_s...     30
3   frontend-2    container.node-5.container_cpu_load_average_10s     30
4   frontend-2      container.node-5.container_cpu_system_seconds     30
..         ...                                                ...    ...
45  frontend-2  container.node-5.container_network_transmit_er...     30
46  frontend-2  container.node-5.container_network_transmit_pa...     30
47  frontend-2  container.node-5.container_network_transmit_pa...     30
48  frontend-2                 container.node-5.container_sockets     30
49  frontend-2         container.node-5.container_spec_cpu_period     30

[50 rows x 3 columns],                     cmdb_id                                           kpi_name  count
0   productcatalogservice-0         container.node-5.container_cpu_cfs_periods     30
1   productcatalogservice-0  container.node-5.container_cpu_cfs_throttled_p...     30
2   productcatalogservice-0  container.node-5.container_cpu_cfs_throttled_s...     30
3   productcatalogservice-0    container.node-5.container_cpu_load_average_10s     30
4   productcatalogservice-0      container.node-5.container_cpu_system_seconds     30
..                      ...                                                ...    ...
45  productcatalogservice-0  container.node-5.container_network_transmit_er...     30
46  productcatalogservice-0  container.node-5.container_network_transmit_pa...     30
47  productcatalogservice-0  container.node-5.container_network_transmit_pa...     30
48  productcatalogservice-0                 container.node-5.container_sockets     30
49  productcatalogservice-0         container.node-5.container_spec_cpu_period     30

[50 rows x 3 columns],                     cmdb_id                                           kpi_name  count
0   productcatalogservice-1         container.node-5.container_cpu_cfs_periods     30
1   productcatalogservice-1  container.node-5.container_cpu_cfs_throttled_p...     30
2   productcatalogservice-1  container.node-5.container_cpu_cfs_throttled_s...     30
3   productcatalogservice-1    container.node-5.container_cpu_load_average_10s     30
4   productcatalogservice-1      container.node-5.container_cpu_system_seconds     30
..                      ...                                                ...    ...
45  productcatalogservice-1  container.node-5.container_network_transmit_er...     30
46  productcatalogservice-1  container.node-5.container_network_transmit_pa...     30
47  productcatalogservice-1  container.node-5.container_network_transmit_pa...     30
48  productcatalogservice-1                 container.node-5.container_sockets     30
49  productcatalogservice-1         container.node-5.container_spec_cpu_period     30

[50 rows x 3 columns],                     cmdb_id                                           kpi_name  count
0   productcatalogservice-2         container.node-5.container_cpu_cfs_periods     30
1   productcatalogservice-2  container.node-5.container_cpu_cfs_throttled_p...     30
2   productcatalogservice-2  container.node-5.container_cpu_cfs_throttled_s...     30
3   productcatalogservice-2    container.node-5.container_cpu_load_average_10s     30
4   productcatalogservice-2      container.node-5.container_cpu_system_seconds     30
..                      ...                                                ...    ...
45  productcatalogservice-2  container.node-5.container_network_transmit_er...     30
46  productcatalogservice-2  container.node-5.container_network_transmit_pa...     30
47  productcatalogservice-2  container.node-5.container_network_transmit_pa...     30
48  productcatalogservice-2                 container.node-5.container_sockets     30
49  productcatalogservice-2         container.node-5.container_spec_cpu_period     30

[50 rows x 3 columns],               cmdb_id                                           kpi_name  count
0   currencyservice-1         container.node-5.container_cpu_cfs_periods     30
1   currencyservice-1  container.node-5.container_cpu_cfs_throttled_p...     30
2   currencyservice-1  container.node-5.container_cpu_cfs_throttled_s...     30
3   currencyservice-1    container.node-5.container_cpu_load_average_10s     30
4   currencyservice-1      container.node-5.container_cpu_system_seconds     30
..                ...                                                ...    ...
45  currencyservice-1  container.node-5.container_network_transmit_er...     30
46  currencyservice-1  container.node-5.container_network_transmit_pa...     30
47  currencyservice-1  container.node-5.container_network_transmit_pa...     30
48  currencyservice-1                 container.node-5.container_sockets     30
49  currencyservice-1         container.node-5.container_spec_cpu_period     30

[50 rows x 3 columns],        cmdb_id                                           kpi_name  count
0   frontend-0         container.node-5.container_cpu_cfs_periods     30
1   frontend-0  container.node-5.container_cpu_cfs_throttled_p...     30
2   frontend-0  container.node-5.container_cpu_cfs_throttled_s...     30
3   frontend-0    container.node-5.container_cpu_load_average_10s     30
4   frontend-0      container.node-5.container_cpu_system_seconds     30
..         ...                                                ...    ...
45  frontend-0  container.node-5.container_network_transmit_er...     30
46  frontend-0  container.node-5.container_network_transmit_pa...     30
47  frontend-0  container.node-5.container_network_transmit_pa...     30
48  frontend-0                 container.node-5.container_sockets     30
49  frontend-0         container.node-5.container_spec_cpu_period     30

[50 rows x 3 columns],                    cmdb_id  distinct_kpi_names
0        currencyservice-1                  76
1               frontend-0                 180
2               frontend-2                 180
3  productcatalogservice-0                 116
4  productcatalogservice-1                 116
5  productcatalogservice-2                 116)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

# Reuse df_metric and tz if available
tz = pytz.timezone('Asia/Shanghai')
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure ts_dt and ts_min exist and are timezone-aware in Asia/Shanghai
if 'ts_dt' not in df_metric.columns:
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and convert to Asia/Shanghai
    if df_metric['ts_dt'].dt.tz is None:
        df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df_metric['ts_dt'] = df_metric['ts_dt'].dt.tz_convert(tz)
if 'ts_min' not in df_metric.columns:
    df_metric['ts_min'] = df_metric['ts_dt'].dt.floor('T')

# Target KPI list (exact strings)
targets = [
    ('frontend-2', 'container.node-5.container_memory_cache'),
    ('frontend-2', 'container.node-5.container_fs_writes_MB./dev/vda'),
    ('frontend-0', 'container.node-5.container_memory_rss'),
    ('productcatalogservice-0', 'container.node-5.container_memory_rss'),
    ('productcatalogservice-1', 'container.node-5.container_memory_rss'),
    ('productcatalogservice-2', 'container.node-5.container_memory_rss'),
    ('currencyservice-1', 'container.node-5.container_memory_rss'),
    ('currencyservice2-0', 'container.node-6.container_memory_rss')
]

# Compute global p95 and p5 for all (cmdb_id, kpi_name) (FULL series)
thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], sort=False)['value']
    .agg(p95=lambda x: float(x.quantile(0.95)), p5=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# Incident window (inclusive)
start = tz.localize(pd.Timestamp('2022-03-21 21:30:00'))
end = tz.localize(pd.Timestamp('2022-03-21 21:59:00'))

rows = []
for cmdb_id, kpi_name in targets:
    # get thresholds for this pair
    th = thresholds[(thresholds['cmdb_id']==cmdb_id) & (thresholds['kpi_name']==kpi_name)]
    if th.empty:
        # no data for this pair
        continue
    p95 = float(th['p95'].iloc[0])
    p5 = float(th['p5'].iloc[0])
    # Filter full window rows for this pair
    df_pair = df_metric[
        (df_metric['cmdb_id']==cmdb_id) &
        (df_metric['kpi_name']==kpi_name) &
        (df_metric['ts_min'] >= start) &
        (df_metric['ts_min'] <= end)
    ].sort_values('ts_min').copy()
    if df_pair.empty:
        continue
    # Determine anomaly type per row using global thresholds (high takes precedence if both)
    df_pair['is_high'] = df_pair['value'] >= p95
    df_pair['is_low'] = df_pair['value'] <= p5
    # Keep only anomaly rows
    df_anom = df_pair[(df_pair['is_high']) | (df_pair['is_low'])].copy()
    if df_anom.empty:
        continue
    # Determine anomaly_direction for each anomaly row (prefer 'high' when both true)
    df_anom['anomaly_type'] = np.where(df_anom['is_high'], 'high', 'low')
    # Compute time difference in minutes between successive anomaly rows
    df_anom = df_anom.sort_values('ts_min').reset_index(drop=True)
    df_anom['dt_min'] = df_anom['ts_min'].diff().dt.total_seconds().div(60).fillna(9999)
    # Start a new segment when dt_min != 1 or anomaly_type changed from previous
    df_anom['prev_type'] = df_anom['anomaly_type'].shift(1).fillna('')
    df_anom['new_seg'] = (df_anom['dt_min'] != 1) | (df_anom['anomaly_type'] != df_anom['prev_type'])
    df_anom['seg_id'] = df_anom['new_seg'].cumsum()
    # Group segments and pick earliest segment (min ts)
    segs = df_anom.groupby('seg_id', sort=False)
    # Select earliest segment by smallest start timestamp
    seg_starts = segs['ts_min'].min().reset_index().sort_values('ts_min')
    first_seg_id = seg_starts['seg_id'].iloc[0]
    seg_rows = df_anom[df_anom['seg_id']==first_seg_id].copy()
    seg_start_ts = seg_rows['ts_min'].min()
    seg_end_ts = seg_rows['ts_min'].max()
    seg_len = int(seg_rows.shape[0])
    value_at_start = float(seg_rows.loc[seg_rows['ts_min']==seg_start_ts, 'value'].iloc[0])
    max_in_seg = float(seg_rows['value'].max())
    min_in_seg = float(seg_rows['value'].min())
    seg_type = seg_rows['anomaly_type'].iloc[0]
    breached_threshold = p95 if seg_type == 'high' else p5
    absolute_severity = abs(value_at_start - breached_threshold)
    rows.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'p95': round(p95,6),
        'p5': round(p5,6),
        'anomaly_type': seg_type,
        'segment_start_timestamp': seg_start_ts.isoformat(),
        'segment_end_timestamp': seg_end_ts.isoformat(),
        'segment_length_minutes': seg_len,
        'value_at_segment_start': round(value_at_start,6),
        'max_value_in_segment': round(max_in_seg,6),
        'min_value_in_segment': round(min_in_seg,6),
        'absolute_severity': round(float(absolute_severity),6)
    })

# Build result DataFrame (one row per target found, up to len(targets))
result_df = pd.DataFrame(rows, columns=[
    'cmdb_id','kpi_name','p95','p5','anomaly_type',
    'segment_start_timestamp','segment_end_timestamp','segment_length_minutes',
    'value_at_segment_start','max_value_in_segment','min_value_in_segment','absolute_severity'
])

# Keep result compact and limited to one row per target (already ensured)
result_df

```
Out[10]:
```
```python
# Summarize the detected consecutive anomaly segments for the requested KPI series.
try:
    result_df
except NameError:
    summary = "Anomaly segment results are not available in memory. Please re-run the segment detection step."
else:
    rows = []
    rows.append("Anomaly segments found in window 2022-03-21 21:30–21:59 (UTC+8):")
    for _, r in result_df.iterrows():
        cm = r['cmdb_id']
        kpi = r['kpi_name']
        typ = r['anomaly_type']
        start = r['segment_start_timestamp']
        end = r['segment_end_timestamp']
        length = int(r['segment_length_minutes'])
        val0 = float(r['value_at_segment_start'])
        vmax = float(r['max_value_in_segment'])
        vmin = float(r['min_value_in_segment'])
        p95 = float(r['p95'])
        p5 = float(r['p5'])
        sev = float(r['absolute_severity'])
        rows.append(
            f"- {cm} | {kpi} | type={typ} | start={start} | end={end} | len_min={length} | "
            f"start_val={val0:.6g} | max={vmax:.6g} | min={vmin:.6g} | p95={p95:.6g} | p5={p5:.6g} | severity={sev:.6g}"
        )
    rows.append("")
    rows.append("Key takeaways:")
    rows.append("- frontend-2.container_memory_cache: a high segment starting 21:32 lasting 2 minutes with ~1.96e6 above P95 (large memory_cache increase).")
    rows.append("- frontend-2.fs_writes: detected a short low segment at 21:30–21:31 but values were zero (no magnitude severity).")
    rows.append("- frontend-0 and productcatalogservice-0 show short low-side RSS anomalies at window start (single-minute segments).")
    rows.append("- productcatalogservice-1 and productcatalogservice-2 show single-minute high RSS anomalies (one at 21:32, one at 21:38).")
    rows.append("- currencyservice-1 shows a low RSS anomaly at 21:30 (single minute); currencyservice2-0 shows a 2-minute high RSS segment around 21:33–21:34.")
    rows.append("")
    rows.append("Recommendation: prioritize investigation of the memory-related high segments (frontend-2 memory_cache, currencyservice2-0 and productcatalogservice RSS spikes) and correlate with frontend traffic peaks at ~21:32–21:34 UTC+8; ignore constant-zero fs_writes low entry as config/zero-value.")
    summary = "\n".join(rows)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                          kpi_name           p95          p5 anomaly_type    segment_start_timestamp      segment_end_timestamp  segment_length_minutes  value_at_segment_start  max_value_in_segment  min_value_in_segment  absolute_severity
0               frontend-2           container.node-5.container_memory_cache  2.140492e+08  22433792.0         high  2022-03-21T21:32:00+08:00  2022-03-21T21:33:00+08:00                       2             216010752.0           216010752.0           214716416.0          1961574.4
1               frontend-2  container.node-5.container_fs_writes_MB./dev/vda  1.503900e-02         0.0          low  2022-03-21T21:30:00+08:00  2022-03-21T21:31:00+08:00                       2                     0.0                   0.0                   0.0                0.0
2               frontend-0             container.node-5.container_memory_rss  1.918403e+07  18688307.2          low  2022-03-21T21:31:00+08:00  2022-03-21T21:31:00+08:00                       1              18661376.0            18661376.0            18661376.0            26931.2
3  productcatalogservice-0             container.node-5.container_memory_rss  1.522790e+07  14823321.6          low  2022-03-21T21:30:00+08:00  2022-03-21T21:30:00+08:00                       1              14813184.0            14813184.0            14813184.0            10137.6
4  productcatalogservice-1             container.node-5.container_memory_rss  1.649992e+07  16122163.2         high  2022-03-21T21:32:00+08:00  2022-03-21T21:32:00+08:00                       1              16627712.0            16627712.0            16627712.0           127795.2
5  productcatalogservice-2             container.node-5.container_memory_rss  1.516237e+07  14637977.6         high  2022-03-21T21:38:00+08:00  2022-03-21T21:38:00+08:00                       1              15171584.0            15171584.0            15171584.0             9216.0
6        currencyservice-1             container.node-5.container_memory_rss  3.546388e+07  34881638.4          low  2022-03-21T21:30:00+08:00  2022-03-21T21:30:00+08:00                       1              34764800.0            34764800.0            34764800.0           116838.4
7       currencyservice2-0             container.node-6.container_memory_rss  3.242793e+07  31486464.0         high  2022-03-21T21:33:00+08:00  2022-03-21T21:34:00+08:00                       2              32473088.0            32546816.0            32473088.0            45158.4```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing dataframes if present, otherwise load
tz = pytz.timezone('Asia/Shanghai')
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure datetime columns (timezone-aware Asia/Shanghai) and minute-floor
for df in (df_metric, df_trace):
    if 'ts_dt' not in df.columns:
        df['ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        if df['ts_dt'].dt.tz is None:
            df['ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        else:
            df['ts_dt'] = df['ts_dt'].dt.tz_convert(tz)
    df['ts_min'] = df['ts_dt'].dt.floor('T')

# Compute global thresholds (P95/P5) on the FULL series if not already computed
try:
    thresholds_metric
except NameError:
    thresholds_metric = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], sort=False)['value']
        .agg(p95=lambda x: float(x.quantile(0.95)), p5=lambda x: float(x.quantile(0.05)))
        .reset_index()
    )

try:
    thresholds_trace
except NameError:
    thresholds_trace = (
        df_trace
        .groupby(['cmdb_id', 'trace_name'], sort=False)['value']
        .agg(p95=lambda x: float(x.quantile(0.95)), p5=lambda x: float(x.quantile(0.05)))
        .reset_index()
    )

# Target list (exact names)
targets = [
    ('metric', 'currencyservice-1', 'container.node-5.container_memory_rss'),
    ('metric', 'frontend-0', 'container.node-5.container_memory_rss'),
    ('metric', 'currencyservice2-0', 'container.node-6.container_memory_rss'),
    ('metric', 'productcatalogservice-2', 'container.node-5.container_memory_rss'),
    ('metric', 'productcatalogservice-1', 'container.node-5.container_memory_rss'),
    ('trace', 'productcatalogservice-2', 'trace.from_frontend-1.row_count'),
    ('trace', 'productcatalogservice-2', 'trace.from_frontend-2.row_count'),
    ('trace', 'productcatalogservice-1', 'trace.from_frontend-1.row_count'),
]

# Incident window
start = tz.localize(pd.Timestamp('2022-03-21 21:30:00'))
end = tz.localize(pd.Timestamp('2022-03-21 21:59:00'))
check_time = tz.localize(pd.Timestamp('2022-03-21 21:34:00'))

out_rows = []
for series_type, cmdb_id, series_name in targets:
    if series_type == 'metric':
        th = thresholds_metric[(thresholds_metric['cmdb_id']==cmdb_id) & (thresholds_metric['kpi_name']==series_name)]
        df_source = df_metric
        name_col = 'kpi_name'
    else:
        th = thresholds_trace[(thresholds_trace['cmdb_id']==cmdb_id) & (thresholds_trace['trace_name']==series_name)]
        df_source = df_trace
        name_col = 'trace_name'
    if th.empty:
        # no threshold info
        out_rows.append({
            'series_type': series_type,
            'cmdb_id': cmdb_id,
            'series_name': series_name,
            'p95': np.nan,
            'p5': np.nan,
            'earliest_anomaly_timestamp_in_window': '',
            'anomaly_direction_at_earliest': '',
            'value_at_earliest': np.nan,
            'segment_includes_2022-03-21T21:34:00+08:00': False,
            'abs_diff_at_earliest': np.nan
        })
        continue
    p95 = float(th['p95'].iloc[0])
    p5 = float(th['p5'].iloc[0])
    # filter to pair and window
    df_pair = df_source[
        (df_source['cmdb_id']==cmdb_id) &
        (df_source[name_col]==series_name) &
        (df_source['ts_min'] >= start) &
        (df_source['ts_min'] <= end)
    ].sort_values('ts_min').copy()
    if df_pair.empty:
        out_rows.append({
            'series_type': series_type,
            'cmdb_id': cmdb_id,
            'series_name': series_name,
            'p95': round(p95,6),
            'p5': round(p5,6),
            'earliest_anomaly_timestamp_in_window': '',
            'anomaly_direction_at_earliest': '',
            'value_at_earliest': np.nan,
            'segment_includes_2022-03-21T21:34:00+08:00': False,
            'abs_diff_at_earliest': np.nan
        })
        continue
    # detect anomalies in-window
    df_pair['is_high'] = df_pair['value'] >= p95
    df_pair['is_low'] = df_pair['value'] <= p5
    df_anom = df_pair[(df_pair['is_high']) | (df_pair['is_low'])].copy()
    if df_anom.empty:
        out_rows.append({
            'series_type': series_type,
            'cmdb_id': cmdb_id,
            'series_name': series_name,
            'p95': round(p95,6),
            'p5': round(p5,6),
            'earliest_anomaly_timestamp_in_window': '',
            'anomaly_direction_at_earliest': '',
            'value_at_earliest': np.nan,
            'segment_includes_2022-03-21T21:34:00+08:00': False,
            'abs_diff_at_earliest': np.nan
        })
        continue
    # earliest anomaly row
    df_anom = df_anom.sort_values('ts_min').reset_index(drop=True)
    earliest_idx = 0
    earliest_row = df_anom.loc[earliest_idx]
    earliest_ts = earliest_row['ts_min']
    # determine direction at earliest (prefer high)
    if earliest_row['is_high']:
        dir_ear = 'high'
        abs_diff = abs(float(earliest_row['value']) - p95)
        threshold_val = p95
    else:
        dir_ear = 'low'
        abs_diff = abs(float(earliest_row['value']) - p5)
        threshold_val = p5
    value_at_earliest = float(earliest_row['value'])
    # find consecutive anomaly segment that contains the earliest anomaly
    # compute dt and segment ids across df_anom, grouping when gap != 1 minute or direction changes
    df_anom['anomaly_type'] = np.where(df_anom['is_high'], 'high', 'low')
    df_anom['dt_min'] = df_anom['ts_min'].diff().dt.total_seconds().div(60).fillna(9999)
    df_anom['prev_type'] = df_anom['anomaly_type'].shift(1).fillna('')
    df_anom['new_seg'] = (df_anom['dt_min'] != 1) | (df_anom['anomaly_type'] != df_anom['prev_type'])
    df_anom['seg_id'] = df_anom['new_seg'].cumsum()
    seg_id_of_earliest = int(df_anom.loc[earliest_idx, 'seg_id'])
    seg_rows = df_anom[df_anom['seg_id']==seg_id_of_earliest]
    seg_start = seg_rows['ts_min'].min()
    seg_end = seg_rows['ts_min'].max()
    seg_len = int(seg_rows.shape[0])
    seg_max = float(seg_rows['value'].max())
    seg_min = float(seg_rows['value'].min())
    includes_check = (seg_start <= check_time) and (check_time <= seg_end)
    out_rows.append({
        'series_type': series_type,
        'cmdb_id': cmdb_id,
        'series_name': series_name,
        'p95': round(p95,6),
        'p5': round(p5,6),
        'earliest_anomaly_timestamp_in_window': seg_rows['ts_min'].min().isoformat() if not seg_rows.empty else '',
        'anomaly_direction_at_earliest': dir_ear,
        'value_at_earliest': round(value_at_earliest,6),
        'segment_includes_2022-03-21T21:34:00+08:00': bool(includes_check),
        'abs_diff_at_earliest': round(float(abs_diff),6)
    })

# Build result DataFrame and keep columns specified, up to 8 rows
result_df = pd.DataFrame(out_rows, columns=[
    'series_type','cmdb_id','series_name','p95','p5',
    'earliest_anomaly_timestamp_in_window','anomaly_direction_at_earliest',
    'value_at_earliest','segment_includes_2022-03-21T21:34:00+08:00','abs_diff_at_earliest'
])

# Display compact result
result_df

```
Out[11]:
```
```python
summary = (
    "Earliest anomalies in window 2022-03-21 21:30–21:59 (UTC+8) for requested series:\n\n"
    "1) metric, currencyservice-1, container.node-5.container_memory_rss: p95=35463884.80 p5=34881638.40 | earliest=2022-03-21T21:30:00+08:00 | direction=low | value=34764800.0 | includes_21:34=False | abs_diff=116838.40\n"
    "2) metric, frontend-0, container.node-5.container_memory_rss: p95=19184025.60 p5=18688307.20 | earliest=2022-03-21T21:31:00+08:00 | direction=low | value=18661376.0 | includes_21:34=False | abs_diff=26931.20\n"
    "3) metric, currencyservice2-0, container.node-6.container_memory_rss: p95=32427929.60 p5=31486464.00 | earliest=2022-03-21T21:33:00+08:00 | direction=high | value=32473088.0 | includes_21:34=True | abs_diff=45158.40\n"
    "4) metric, productcatalogservice-2, container.node-5.container_memory_rss: p95=15162368.00 p5=14637977.60 | earliest=2022-03-21T21:38:00+08:00 | direction=high | value=15171584.0 | includes_21:34=False | abs_diff=9216.00\n"
    "5) metric, productcatalogservice-1, container.node-5.container_memory_rss: p95=16499916.80 p5=16122163.20 | earliest=2022-03-21T21:32:00+08:00 | direction=high | value=16627712.0 | includes_21:34=False | abs_diff=127795.20\n\n"
    "6) trace, productcatalogservice-2, trace.from_frontend-1.row_count: p95=91.20 p5=50.15 | earliest=2022-03-21T21:30:00+08:00 | direction=high | value=93.0 | includes_21:34=False | abs_diff=1.80\n"
    "7) trace, productcatalogservice-2, trace.from_frontend-2.row_count: p95=117.65 p5=39.45 | earliest=2022-03-21T21:32:00+08:00 | direction=high | value=129.0 | includes_21:34=False | abs_diff=11.35\n"
    "8) trace, productcatalogservice-1, trace.from_frontend-1.row_count: p95=92.20 p5=50.15 | earliest=2022-03-21T21:30:00+08:00 | direction=high | value=94.0 | includes_21:34=False | abs_diff=1.80\n\n"
    "Key takeaways:\n"
    "- Only currencyservice2-0.metric (container_memory_rss) has an earliest anomaly segment that includes 2022-03-21T21:34:00+08:00 (a 2-minute high segment starting 21:33).\n"
    "- Other metric anomalies are mostly single-minute segments (productcatalogservice and frontend) or occur earlier/later and do not include 21:34.\n"
    "- Trace anomalies (productcatalogservice from_frontend row_count) show early high counts (21:30 and 21:32) but their earliest segments do not include 21:34.\n\n"
    "Recommendation:\n"
    "- Prioritize investigation of currencyservice2-0 around 21:33–21:34 (memory_rss high segment). Also correlate the productcatalogservice/frontend trace spikes (21:30–21:32) with metric memory/RSS spikes to find potential traffic/memory relationships."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

series_type                  cmdb_id                            series_name          p95           p5 earliest_anomaly_timestamp_in_window anomaly_direction_at_earliest  value_at_earliest  segment_includes_2022-03-21T21:34:00+08:00  abs_diff_at_earliest
0      metric        currencyservice-1  container.node-5.container_memory_rss  35463884.80  34881638.40            2022-03-21T21:30:00+08:00                           low         34764800.0                                       False             116838.40
1      metric               frontend-0  container.node-5.container_memory_rss  19184025.60  18688307.20            2022-03-21T21:31:00+08:00                           low         18661376.0                                       False              26931.20
2      metric       currencyservice2-0  container.node-6.container_memory_rss  32427929.60  31486464.00            2022-03-21T21:33:00+08:00                          high         32473088.0                                        True              45158.40
3      metric  productcatalogservice-2  container.node-5.container_memory_rss  15162368.00  14637977.60            2022-03-21T21:38:00+08:00                          high         15171584.0                                       False               9216.00
4      metric  productcatalogservice-1  container.node-5.container_memory_rss  16499916.80  16122163.20            2022-03-21T21:32:00+08:00                          high         16627712.0                                       False             127795.20
5       trace  productcatalogservice-2        trace.from_frontend-1.row_count        91.20        50.15            2022-03-21T21:30:00+08:00                          high               93.0                                       False                  1.80
6       trace  productcatalogservice-2        trace.from_frontend-2.row_count       117.65        39.45            2022-03-21T21:32:00+08:00                          high              129.0                                       False                 11.35
7       trace  productcatalogservice-1        trace.from_frontend-1.row_count        92.20        50.15            2022-03-21T21:30:00+08:00                          high               94.0                                       False                  1.80```
```